# General Backtester for ATM Panoptions 
On:
- Any Uni V3 Pool
- Any Time Period
- Any Range Factor
- Any # of put & call legs
- Any delta for each leg
- Includes Subsample Testing
- Support for custom Base data
- Added yearly rolling (previously had daily, weekly, and monthly)

### Import Python Packages

In [1]:
import pandas as pd
import pandas_gbq
import pydata_google_auth
import numpy as np
import matplotlib.pyplot as plt
import math
import calendar

from tqdm import tqdm
from datetime import timedelta
from functools import reduce
pd.options.mode.chained_assignment = None

In [35]:
class LP_Rebalance:
    def __init__(
        self,
        token_0: str,
        token_1: str,
        fee: int, # in bps
        local_dir: str,
        start_t: str,
        end_t: str,
        range_perc: float, # i.e. +/- 30%
        col_ratio: float=100, # i.e. 100%
        com_ratio: float=0, # i.e. 0bps
        PLP_annual_yield: float=0, # i.e. 0%
        spread_mult: float=1,
        pool_data: pd.DataFrame=None,
        inverse_price: bool=True,
        legs: dict={'short_put': 1},
        delta: dict={'short_put': 0.5},
        chain: str='Ethereum',
    ) -> None:
        """
        Initialize LP Rebalancing class
        
        :token_0 Token 0
        :token_1 Token 1
        :fee fee rate in bps
        :local_dir local directory to save plots
        :start_t start time (inclusive) (i.e. '2021-05-06 00:06:12' or '2021-05-06')
        :end_t end time (exclusive)
        :range_perc range percent (for +/- 30%, input 30)
        :col_ratio collateral ratio (%) - assumes position is never underwater or liquidated
        :com_ratio commission ratio (bps)
        :PLP_annual_yield annual yield from providing Panoptic liquidity (%)
        :spread_mult spread multiplier for buying popular options
        :pool_data enter dataframe of pool data from another strategy to save load time. Default=None
        :inverse_price invert the price (if true: gets price of token_1 in terms of token_0)
        :legs type of option ['SHORT_PUT', 'LONG_PUT', 'SHORT_CALL', 'LONG_CALL'] and quantity
        """
        self.token_0 = token_0
        self.token_1 = token_1
        self.fee_rate = fee / 10_000
        self.pool_address = LP_Rebalance.get_pool_address(self.token_0, self.token_1, fee, chain)
        self.dec_0 = LP_Rebalance.get_token_dec(self.token_0)
        self.dec_1 = LP_Rebalance.get_token_dec(self.token_1)
        self.tick_spacing = LP_Rebalance.get_tick_spacing(fee)
        ''' Our_capital is 1 b/c it allows results to be interpreted as percentages
        Strategy does accounting in numeraire token
        I.e. start with 1 token of numeraire, split 50/50 to LP.
        At end of rebalancing period (e.g. 1 day), we sell everything back into the numeraire token
        And calculate the % change (return) on the numeraire token '''
        self.our_capital = 1 # in token y
        self.raw_dir = local_dir
        self.start_t = start_t
        self.end_t = end_t
        self.range_perc = (range_perc / 100) + 1
        self.tick_width = LP_Rebalance.convert_r_to_tick_width(self.range_perc)
        self.col_ratio = col_ratio / 100
        self.com_ratio = com_ratio / 10_000
        self.collateral = self.our_capital * self.col_ratio
        self.PLP_annual_yield = PLP_annual_yield / 100
        self.spread_mult = spread_mult
        ''' If inverse_price, our numeraire is token_0
        Else, our numeraire is token_1
        Ex: For USDC/ETH pool:
        token 0 = USDC
        token 1 = ETH
        Inverse_price = True -> we calculate returns in terms of USDC
        Inverse_price = False -> we calculate returns in terms of ETH
        This will affect results! '''
        self.inverse_price = inverse_price

        valid_options = ['SHORT_PUT', 'LONG_PUT', 'SHORT_CALL', 'LONG_CALL']
        for leg in legs:
            if leg not in valid_options:
                raise BaseException(f"Invalid option type. Valid types: {valid_options}")
        self.legs = legs
        self.delta = delta

        self.BASE = 1.0001

        if pool_data is None:
            self.load_pool_data()
        else:
            self.data = pool_data
            if 'date' not in self.data.columns: # data loaded in from other blockchains like Base
                self.data = self.data[
                    (self.data["block_timestamp"] >= self.start_t) & 
                    (self.data["block_timestamp"] <= self.end_t) &
                    (self.data["address"] == self.pool_address)
                ]
                self.transform_pool_data(indexSupply=True)

        

    def load_pool_data(self):
        """Loads Univ3 pool swap data"""
        # Replace this function with your own GBQ data fetcher
        # See: https://github.com/panoptic-labs/research/blob/JP/_research-bites/DataTutorial/tutorial.ipynb
        print("Loading Data...")

        SCOPES = [
            'https://www.googleapis.com/auth/cloud-platform',
            'https://www.googleapis.com/auth/drive',
        ]

        credentials = pydata_google_auth.get_user_credentials(
            SCOPES,
            auth_local_webserver=True,
        )

        query = f"""
        SELECT DISTINCT *
        FROM `arcane-world-371019.First_sync.1`
        WHERE address = '{self.pool_address}'
            AND block_timestamp >= '{self.start_t}'
            AND block_timestamp < '{self.end_t}'
        ORDER BY block_number, transaction_index
        """
        self.data = pandas_gbq.read_gbq(query, project_id = "arcane-world-371019", credentials=credentials)
        self.transform_pool_data()

    def transform_pool_data(self, indexSupply=False):
        if not indexSupply:
            """Transform amounts to human-readable format"""
            self.data['amount0'] = self.data['amount0'].apply(LP_Rebalance.get_twos_comp)
            self.data['amount1'] = self.data['amount1'].apply(LP_Rebalance.get_twos_comp)

            # Unpack sqrt price
            self.data['sqrtPrice'] = self.data['sqrtPrice'].apply(LP_Rebalance.unpack_sqrtprice)

        else:
            self.data['sqrtPrice'] = self.data['sqrtPrice'].astype(str).astype(float).apply(
                lambda x: LP_Rebalance.unpack_sqrtprice(x, is_hex=False)
            )
        
        self.data['amount0'] = self.data['amount0'].astype(float) / (10 ** self.dec_0)
        self.data['amount1'] = self.data['amount1'].astype(float) / (10 ** self.dec_1)

        self.data['price'] = self.data['tick'].apply(self.convert_tick)

        # Note our GBQ query ordered by blocknumber, transaciton_index
        self.data.set_index('block_timestamp', inplace=True) # Assume block timestamps are accurate "enough" for our use-case
        self.data.index = pd.to_datetime(self.data.index)

        # Calculate sqrt price change
        self.data = LP_Rebalance.calc_sqrtprice_change(self.data)

        self.data['date'] = self.data.index.date
        # Lag prices (Have to do for illiquid pools w/few txs per day)
        self.data['price_lag'] = self.data['price'].shift()
        self.data['tick_lag'] = self.data['tick'].shift()
        self.data['sqrtPrice_lag'] = self.data['sqrtPrice'].shift()
        self.data = self.data.iloc[1:]

    def run_strat(self, rolling_frequency: str='a'):
        print("Running Strategy")
        self.create_strat(rolling_frequency=rolling_frequency)
        quantity_total = sum(self.legs.values())

        if rolling_frequency == 'a' or rolling_frequency == 'd':
            tqdm().pandas(desc='Step 1') # Calculate liquidity per tick
            self.daily['liq_per_tick'] = self.daily.progress_apply(self.calc_liq_per_tick, axis=1)        
            tqdm().pandas(desc='Step 2') # Calculate fees
            self.daily[['our_fee', 'fee_asset']] = self.daily.progress_apply(self.calc_fees, axis=1)
            self.daily_fees_legs = {}
            for leg, quantity in self.legs.items():
                self.daily_fees_legs[leg] = self.daily.groupby('date').apply(self.get_fee_summary, leg) * quantity
            self.daily_fees = reduce(lambda x, y: x.add(y, fill_value=0), self.daily_fees_legs.values())
            self.daily_fees.rename(columns={0: 'fees_total'}, inplace=True)
            self.daily_fees['fees_perc'] = self.daily_fees['fees_total'] / (self.collateral * quantity_total)
            tqdm().pandas(desc='Step 3') # Calculate PnL
            self.daily_pos_legs = {}
            for i, (leg, quantity) in enumerate(self.legs.items()):
                self.daily_pos_legs[leg] = self.daily.groupby('date').progress_apply(self.calc_position, option_type=leg, leg_number=i) * quantity
            self.daily_pos = reduce(lambda x, y: x.add(y, fill_value=0), self.daily_pos_legs.values())
            # Assumes same collateralization for each leg (TODO: allow customizable collateral)
            self.daily_pos['pnl_perc'] = self.daily_pos['pnl'] / (self.collateral * quantity_total)

        if rolling_frequency == 'a' or rolling_frequency == 'w':
            tqdm().pandas(desc='Step 1')
            self.weekly['liq_per_tick'] = self.weekly.progress_apply(self.calc_liq_per_tick, axis=1)
            tqdm().pandas(desc='Step 2')   
            self.weekly[['our_fee', 'fee_asset']] = self.weekly.progress_apply(self.calc_fees, axis=1)
            self.weekly_fees_legs = {}
            for leg, quantity in self.legs.items():
                self.weekly_fees_legs[leg] = self.weekly.groupby([pd.Grouper(level = 'block_timestamp', freq = 'W-MON', closed='left')]).apply(self.get_fee_summary, leg) * quantity
            self.weekly_fees = reduce(lambda x, y: x.add(y, fill_value=0), self.weekly_fees_legs.values())
            self.weekly_fees.rename(columns={0: 'fees_total'}, inplace=True)
            self.weekly_fees['fees_perc'] = self.weekly_fees['fees_total'] / (self.collateral * quantity_total)
            self.weekly_fees.index = self.weekly_fees.index.date
            self.weekly_fees.index.name = 'date'
            tqdm().pandas(desc='Step 3')
            self.weekly_pos_legs = {}
            for i, (leg, quantity) in enumerate(self.legs.items()):
                self.weekly_pos_legs[leg] = self.weekly.groupby([pd.Grouper(level = 'block_timestamp', freq = 'W-MON', closed='left')]).progress_apply(self.calc_position, option_type=leg, leg_number=i) * quantity
            self.weekly_pos = reduce(lambda x, y: x.add(y, fill_value=0), self.weekly_pos_legs.values())
            self.weekly_pos['pnl_perc'] = self.weekly_pos['pnl'] / (self.collateral * quantity_total)
            self.weekly_pos.index = self.weekly_pos.index.date
            self.weekly_pos.index.name = 'date'

        if rolling_frequency == 'a' or rolling_frequency == 'm':
            tqdm().pandas(desc='Step 1')
            self.monthly['liq_per_tick'] = self.monthly.progress_apply(self.calc_liq_per_tick, axis=1)
            tqdm().pandas(desc='Step 2')
            self.monthly[['our_fee', 'fee_asset']] = self.monthly.progress_apply(self.calc_fees, axis=1)
            self.monthly_fees_legs = {}
            for leg, quantity in self.legs.items():
                self.monthly_fees_legs[leg] = self.monthly.groupby(['year', 'month']).apply(self.get_fee_summary, leg) * quantity
            self.monthly_fees = reduce(lambda x, y: x.add(y, fill_value=0), self.monthly_fees_legs.values())
            self.monthly_fees.rename(columns={0: 'fees_total'}, inplace=True)
            self.monthly_fees['fees_perc'] = self.monthly_fees['fees_total'] / (self.collateral * quantity_total)
            y = self.monthly_fees.index.get_level_values('year')
            m = self.monthly_fees.index.get_level_values('month')
            self.monthly_fees.index = pd.to_datetime(y * 10000 + m * 100 + 1, format="%Y%m%d")
            self.monthly_fees.index = [dti.replace(day = calendar.monthrange(dti.year, dti.month)[1]).date() for dti in self.monthly_fees.index]
            self.monthly_fees.index.name = 'date'
            if hasattr(self, 'daily_fees'):
                self.monthly_fees.loc[self.daily_fees.index[0]] = [None, 0]
            self.monthly_fees.index = pd.to_datetime(self.monthly_fees.index)
            self.monthly_fees.sort_index(inplace=True)
            tqdm().pandas(desc='Step 3')
            self.monthly_pos_legs = {}
            for i, (leg, quantity) in enumerate(self.legs.items()):
                self.monthly_pos_legs[leg] = self.monthly.groupby(['year', 'month']).progress_apply(self.calc_position, option_type=leg, leg_number=i) * quantity
            self.monthly_pos = reduce(lambda x, y: x.add(y, fill_value=0), self.monthly_pos_legs.values())
            self.monthly_pos['pnl_perc'] = self.monthly_pos['pnl'] / (self.collateral * quantity_total)

            y = self.monthly_pos.index.get_level_values('year')
            m = self.monthly_pos.index.get_level_values('month')
            self.monthly_pos.index = pd.to_datetime(y * 10000 + m * 100 + 1, format="%Y%m%d")
            self.monthly_pos.index = [dti.replace(day = calendar.monthrange(dti.year, dti.month)[1]).date() for dti in self.monthly_pos.index]
            self.monthly_pos.index.name = 'date'
            if hasattr(self, 'daily_pos'):
                self.monthly_pos.loc[self.daily_pos.index[0]] = [None, None, None, None, None, None, 0]
            self.monthly_pos.index = pd.to_datetime(self.monthly_pos.index)
            self.monthly_pos.sort_index(inplace=True)

        if rolling_frequency == 'a' or rolling_frequency == 'y':
            tqdm().pandas(desc='Step 1')
            self.yearly['liq_per_tick'] = self.yearly.progress_apply(self.calc_liq_per_tick, axis=1)
            tqdm().pandas(desc='Step 2')
            self.yearly[['our_fee', 'fee_asset']] = self.yearly.progress_apply(self.calc_fees, axis=1)
            self.yearly_fees_legs = {}
            for leg, quantity in self.legs.items():
                self.yearly_fees_legs[leg] = self.yearly.groupby(['year']).apply(self.get_fee_summary, leg) * quantity
            self.yearly_fees = reduce(lambda x, y: x.add(y, fill_value=0), self.yearly_fees_legs.values())
            self.yearly_fees.rename(columns={0: 'fees_total'}, inplace=True)
            self.yearly_fees['fees_perc'] = self.yearly_fees['fees_total'] / (self.collateral * quantity_total)
            self.yearly_fees.index = pd.to_datetime(self.yearly_fees.index.astype(str) + '-12-31')
            self.yearly_fees.index.name = 'date'
            if hasattr(self, 'daily_fees') and not self.daily_fees.empty:
                key_ts = pd.to_datetime(self.daily_fees.index[0])
                self.yearly_fees.loc[key_ts] = [None, 0]
            self.yearly_fees.index = pd.to_datetime(self.yearly_fees.index)
            self.yearly_fees.sort_index(inplace=True)
            tqdm().pandas(desc='Step 3')
            self.yearly_pos_legs = {}
            for i, (leg, quantity) in enumerate(self.legs.items()):
                self.yearly_pos_legs[leg] = self.yearly.groupby(['year']).progress_apply(self.calc_position, option_type=leg, leg_number=i) * quantity
            self.yearly_pos = reduce(lambda x, y: x.add(y, fill_value=0), self.yearly_pos_legs.values())
            self.yearly_pos['pnl_perc'] = self.yearly_pos['pnl'] / (self.collateral * quantity_total)
            self.yearly_pos.index = pd.to_datetime(self.yearly_pos.index.astype(str) + '-12-31')
            self.yearly_pos.index.name = 'date'
            if hasattr(self, 'daily_pos'):
                self.yearly_pos.loc[self.daily_pos.index[0]] = [None, None, None, None, None, None, 0]
            self.yearly_pos.index = pd.to_datetime(self.yearly_pos.index)
            self.yearly_pos.sort_index(inplace=True)

        """NEW-oor"""
        if rolling_frequency == 'oor':
            tqdm().pandas(desc='Step 1')
            self.oor['liq_per_tick'] = self.oor.progress_apply(self.calc_liq_per_tick, axis=1)
            tqdm().pandas(desc='Step 2')
            self.oor[['our_fee', 'fee_asset']] = self.oor.progress_apply(self.calc_fees, axis=1)
            self.oor_fees_legs = {}
            for leg, quantity in self.legs.items():
                self.oor_fees_legs[leg] = self.oor.groupby('date').apply(self.get_fee_summary, leg) * quantity
            self.oor_fees = reduce(lambda x, y: x.add(y, fill_value=0), self.oor_fees_legs.values())
            self.oor_fees.rename(columns={0: 'fees_total'}, inplace=True)
            self.oor_fees['fees_perc'] = self.oor_fees['fees_total'] / (self.collateral * quantity_total)
            tqdm().pandas(desc='Step 3')

            self.oor_pos_legs = {}

            for i, (leg, quantity) in enumerate(self.legs.items()):
                leg_daily = (
                    self.oor
                    .groupby('date')
                    .progress_apply(
                        lambda day_df: self._oor_leg_daily_from_segments(
                            day_df, option_type=leg, leg_number=i
                        )
                    )
                )
                leg_daily['pnl'] = leg_daily['pnl'] * quantity

                leg_daily.index = pd.to_datetime(leg_daily.index)
                leg_daily.index.name = 'date'

                self.oor_pos_legs[leg] = leg_daily



            # for i, (leg, quantity) in enumerate(self.legs.items()):
            #     self.oor_pos_legs[leg] = self.oor.groupby('date').progress_apply(self.calc_position, option_type=leg, leg_number=i) * quantity
            self.oor_pos = reduce(lambda x, y: x.add(y, fill_value=0), self.oor_pos_legs.values()) # note that if .oor_pos_legs has 2+ legs, the summed values for x_0, y_0, x, y, pos won't make as much sense
            self.oor_pos['pnl_perc'] = self.oor_pos['pnl'] / (self.collateral * quantity_total)
        """NEW-oor"""

        # Calculate Commissions paid
        self.update_commissions(com_ratio = self.com_ratio * 10_000, rolling_frequency=rolling_frequency)

        # Calculate PLP yield earned
        self.update_PLP_yield(PLP_annual_yield = self.PLP_annual_yield * 100, rolling_frequency=rolling_frequency)

    def _oor_leg_daily_from_segments(self, day_df: pd.DataFrame, option_type: str, leg_number: int) -> pd.Series:
        """
        For a single day (day_df) and a single leg, compute:
        - pnl: sum of PnL over all rebalance segments that touch this day
        - x_0, y_0, x, y, pos: tuples of segment-level values
        """
        reb_col = f"rebalance_id_{leg_number}"

        if day_df.empty:
            return pd.Series({
                'x_0': tuple(),
                'y_0': tuple(),
                'x':   tuple(),
                'y':   tuple(),
                'pos': tuple(),
                'pnl': 0.0,
            })

        x0_list, y0_list, x_list, y_list, pos_list = [], [], [], [], []
        pnl_sum = 0.0

        for reb_id, seg in day_df.groupby(reb_col):
            seg = seg.sort_index().copy()
            seg['price_start'] = seg['price'].iloc[0] # force price_start to be the initial price of the segment to calculate segment's option pnl change marked to market
            res = self.calc_position(seg, option_type=option_type, leg_number=leg_number)

            x0_list.append(float(res['x_0']))
            y0_list.append(float(res['y_0']))
            x_list.append(float(res['x']))
            y_list.append(float(res['y']))
            pos_list.append(float(res['pos']))
            pnl_sum += float(res['pnl'])

        return pd.Series({
            'x_0': tuple(x0_list),
            'y_0': tuple(y0_list),
            'x':   tuple(x_list),
            'y':   tuple(y_list),
            'pos': tuple(pos_list),
            'pnl': pnl_sum,
        })


    def update_commissions(self, com_ratio: float, rolling_frequency: str='a') -> None:
        """
        Update commission ratio

        :com_ratio commission ratio (bps)
        """
        self.com_ratio = com_ratio / 10_000
        coms_total = -1 * self.com_ratio * self.our_capital
        coms_perc = coms_total / self.collateral
        if rolling_frequency == 'a' or rolling_frequency == 'd':
            self.daily_com = pd.DataFrame({'coms_total': coms_total, 'coms_perc': coms_perc}, index=self.daily_pos.index)
        if rolling_frequency == 'a' or rolling_frequency == 'w':
            self.weekly_com = pd.DataFrame({'coms_total': coms_total, 'coms_perc': coms_perc}, index=self.weekly_pos.index)
        if rolling_frequency == 'a' or rolling_frequency == 'm':
            self.monthly_com = pd.DataFrame({'coms_total': coms_total, 'coms_perc': coms_perc}, index=self.monthly_pos.index)
        if rolling_frequency == 'a' or rolling_frequency == 'y':
            self.yearly_com = pd.DataFrame({'coms_total': coms_total, 'coms_perc': coms_perc}, index=self.yearly_pos.index)


    def update_PLP_yield(self, PLP_annual_yield: float, rolling_frequency: str='a') -> None:
        """
        Update Panoptic LP annual yield

        :PLP_annual_yield PLP annual yield (%)
        """
        self.PLP_annual_yield = PLP_annual_yield / 100
        if rolling_frequency == 'a' or rolling_frequency == 'd':
            daily_yield = self.PLP_annual_yield / 365
            self.daily_PLP_yield = pd.DataFrame({'PLP_yield_total': daily_yield * self.collateral, 'PLP_yield_perc': daily_yield}, index=self.daily_pos.index)
        if rolling_frequency == 'a' or rolling_frequency == 'w':
            weekly_yield = self.PLP_annual_yield / 52
            self.weekly_PLP_yield = pd.DataFrame({'PLP_yield_total': weekly_yield * self.collateral, 'PLP_yield_perc': weekly_yield}, index=self.weekly_pos.index)
        if rolling_frequency == 'a' or rolling_frequency == 'm':
            monthly_yield = self.PLP_annual_yield / 12
            self.monthly_PLP_yield = pd.DataFrame({'PLP_yield_total': monthly_yield * self.collateral, 'PLP_yield_perc': monthly_yield}, index=self.monthly_pos.index)
        if rolling_frequency == 'a' or rolling_frequency == 'y':
            yearly_yield = self.PLP_annual_yield
            self.yearly_PLP_yield = pd.DataFrame({'PLP_yield_total': yearly_yield * self.collateral, 'PLP_yield_perc': yearly_yield}, index=self.yearly_pos.index)
        

    def update_spread_mult(self, spread_mult: float) -> None:
        """
        Update spread multiplier

        :spread_mult spread multiplier (e.g. 1.25X)
        """ 
        for fees in [self.daily_fees, self.weekly_fees, self.monthly_fees]:
            fees['fees_total'] *= spread_mult / self.spread_mult
            fees['fees_perc'] = fees['fees_total'] / self.collateral
        self.spread_mult = spread_mult

    def create_strat(self, rolling_frequency: str='a'):
        """
        Define Rebalancing Strategies
        Define 3 strategies: LP +/- X% of start price and rebalance (1) daily, (2) weekly, and (3) monthly
        -Assumes no rebalancing costs in terms of gas fees or slippage/swap fees
        -Assumes no reinvestment/LPing of collected fees
        """

        #TODO: refactor so backtester can choose different deltas for different legs
        long_options = ['LONG_PUT', 'LONG_CALL']
        put_options = ['SHORT_PUT', 'LONG_PUT']

        if rolling_frequency == 'a' or rolling_frequency == 'd':
            daily = self.data.copy()
            daily['price_start'] = daily.groupby('date')['price_lag'].transform(lambda x: x.iloc[0])
            daily['tick_start'] = daily.groupby('date')['tick_lag'].transform(lambda x: x.iloc[0])
            daily['sqrtPrice_start'] = daily.groupby('date')['sqrtPrice_lag'].transform(lambda x: x.iloc[0])
            for i, leg in enumerate(self.delta):
                is_long = True if leg in long_options else False
                is_put = True if leg in put_options else False
                daily[f'strike_start_{i}'] = (
                    daily
                    .groupby('date')['price_start']
                    .transform(
                        lambda group_prices: self.strike_from_delta(
                            spot_price   = group_prices.iloc[0],
                            delta        = self.delta[leg],
                            range_factor = self.range_perc,
                            is_long      = is_long,
                            is_put       = is_put
                        )
                    )
                )
                daily[f'strike_tick_start_{i}'] = (
                    daily
                    .groupby('date')[f'strike_start_{i}']
                    .transform(
                        lambda group_strikes: self.convert_price_to_tick(
                            p=group_strikes.iloc[0]
                        )
                    )
                )
                daily[f'price_a_{i}'] = daily[f'strike_start_{i}'] / self.range_perc
                daily[f'price_b_{i}'] = daily[f'strike_start_{i}'] * self.range_perc
            self.daily = daily

        if rolling_frequency == 'a' or rolling_frequency == 'w':
            weekly = self.data.copy()
            weekly['price_start'] = weekly.groupby([pd.Grouper(level = 'block_timestamp', freq = 'W-MON', closed='left')])['price_lag'].transform(lambda x: x[0])
            weekly['tick_start'] = weekly.groupby([pd.Grouper(level = 'block_timestamp', freq = 'W-MON', closed='left')])['tick_lag'].transform(lambda x: x[0])
            weekly['sqrtPrice_start'] = weekly.groupby([pd.Grouper(level = 'block_timestamp', freq = 'W-MON', closed='left')])['sqrtPrice_lag'].transform(lambda x: x[0])
            for i, leg in enumerate(self.delta):
                is_long = True if leg in long_options else False
                is_put = True if leg in put_options else False
                # Rebalance every Monday (UTC)
                weekly[f'strike_start_{i}'] = (
                    weekly
                    .groupby([pd.Grouper(level='block_timestamp', freq='W-MON', closed='left')])['price_start']
                    .transform(
                        lambda group_prices: self.strike_from_delta(
                            spot_price   = group_prices.iloc[0],
                            delta        = self.delta[leg],
                            range_factor = self.range_perc,
                            is_long      = is_long,
                            is_put       = is_put
                        )
                    )
                )
                weekly[f'strike_tick_start_{i}'] = (
                    weekly
                    .groupby([pd.Grouper(level='block_timestamp', freq='W-MON', closed='left')])[f'strike_start_{i}']
                    .transform(
                        lambda group_strikes: self.convert_price_to_tick(
                            p=group_strikes.iloc[0]
                        )
                    )
                )
                weekly[f'price_a_{i}'] = weekly[f'strike_start_{i}'] / self.range_perc
                weekly[f'price_b_{i}'] = weekly[f'strike_start_{i}'] * self.range_perc
            self.weekly = weekly

        if rolling_frequency == 'a' or rolling_frequency == 'm':
            monthly = self.data.copy()
            monthly['month'] = monthly.index.month
            monthly['year'] = monthly.index.year
            monthly['price_start'] = monthly.groupby(['year', 'month'])['price_lag'].transform(lambda x: x[0])
            monthly['tick_start'] = monthly.groupby(['year', 'month'])['tick_lag'].transform(lambda x: x[0])
            monthly['sqrtPrice_start'] = monthly.groupby(['year', 'month'])['sqrtPrice_lag'].transform(lambda x: x[0])
            for i, leg in enumerate(self.delta):
                is_long = True if leg in long_options else False
                is_put = True if leg in put_options else False
                # Rebalance on the 1st of the month
                monthly[f'strike_start_{i}'] = (
                    monthly
                    .groupby(['year', 'month'])['price_start']
                    .transform(
                        lambda group_prices: self.strike_from_delta(
                            spot_price   = group_prices.iloc[0],  # first spot in this group
                            delta        = self.delta[leg],
                            range_factor = self.range_perc,
                            is_long      = is_long,
                            is_put       = is_put
                        )
                    )
                )
                monthly[f'strike_tick_start_{i}'] = (
                    monthly
                    .groupby(['year', 'month'])[f'strike_start_{i}']
                    .transform(
                        lambda group_strikes: self.convert_price_to_tick(
                            p = group_strikes.iloc[0]
                        )
                    )
                )

                monthly[f'price_a_{i}'] = monthly[f'strike_start_{i}'] / self.range_perc
                monthly[f'price_b_{i}'] = monthly[f'strike_start_{i}'] * self.range_perc
            self.monthly = monthly

        if rolling_frequency == 'a' or rolling_frequency == 'y':
            yearly = self.data.copy()
            yearly['year'] = yearly.index.year
            yearly['price_start'] = yearly.groupby(['year'])['price_lag'].transform(lambda x: x[0])
            yearly['tick_start'] = yearly.groupby(['year'])['tick_lag'].transform(lambda x: x[0])
            yearly['sqrtPrice_start'] = yearly.groupby(['year'])['sqrtPrice_lag'].transform(lambda x: x[0])

            for i, leg in enumerate(self.delta):
                is_long = True if leg in long_options else False
                is_put = True if leg in put_options else False
                yearly[f'strike_start_{i}'] = (
                    yearly
                    .groupby(['year'])['price_start']
                    .transform(
                        lambda group_prices: self.strike_from_delta(
                            spot_price=group_prices.iloc[0],
                            delta=self.delta[leg],
                            range_factor=self.range_perc,
                            is_long=is_long,
                            is_put=is_put,
                        )
                    )
                )
                yearly[f'strike_tick_start_{i}'] = (
                    yearly
                    .groupby(['year'])[f'strike_start_{i}']
                    .transform(lambda group_strikes: self.convert_price_to_tick(group_strikes.iloc[0]))
                )
                yearly[f'price_a_{i}'] = yearly[f'strike_start_{i}'] / self.range_perc
                yearly[f'price_b_{i}'] = yearly[f'strike_start_{i}'] * self.range_perc

            self.yearly = yearly

        """NEW-oor"""
        if rolling_frequency == 'oor':
            df = self.data.copy()
            n = len(df)
            legs = list(self.delta.keys())
            n_legs = len(legs)

            # --- Allocate global (rebalance-start) arrays ---
            price_start_arr     = np.full(n, np.nan, dtype=float)
            tick_start_arr      = np.full(n, np.nan, dtype=float)
            sqrtPrice_start_arr = np.full(n, np.nan, dtype=float)
            rebalance_flag_arr  = np.zeros(n, dtype=bool)

            # --- Allocate per-leg arrays ---
            strike_start_arr      = {i: np.full(n, np.nan, dtype=float) for i in range(n_legs)}
            strike_tick_start_arr = {i: np.full(n, np.nan, dtype=float) for i in range(n_legs)}
            price_a_arr           = {i: np.full(n, np.nan, dtype=float) for i in range(n_legs)}
            price_b_arr           = {i: np.full(n, np.nan, dtype=float) for i in range(n_legs)}
            rebalance_id_arr      = {i: np.zeros(n, dtype=int)          for i in range(n_legs)}

            # --- Stateful current values ---
            current_price_start     = None
            current_tick_start      = None
            current_sqrtPrice_start = None

            current_strike      = {i: None for i in range(n_legs)}
            current_strike_tick = {i: None for i in range(n_legs)}
            current_price_a     = {i: None for i in range(n_legs)}
            current_price_b     = {i: None for i in range(n_legs)}
            rebalance_id        = {i: 0    for i in range(n_legs)}

            for idx in range(n):
                row       = df.iloc[idx]
                price_now = row['price']
                tick_now  = row['tick']
                sqrt_now  = row['sqrtPrice']

                # -----------------------------------------------------------
                # 1. Determine if this row triggers a rebalance
                # -----------------------------------------------------------
                if idx == 0:
                    # First-row initialization
                    do_rebalance = True
                    spot_for_strike = row['price_lag']
                    current_price_start     = row['price_lag']
                    current_tick_start      = row['tick_lag']
                    current_sqrtPrice_start = row['sqrtPrice_lag']

                else:
                    # No automatic daily reset — only react to out-of-range events
                    do_rebalance = False

                    # Check OR condition: ANY leg out of range triggers rebalance
                    for i in range(n_legs):
                        if current_price_a[i] is None:
                            continue
                        if (price_now < current_price_a[i]) or (price_now > current_price_b[i]):
                            do_rebalance = True
                            break

                    if do_rebalance:
                        # Rebalance uses current spot
                        spot_for_strike = price_now
                        current_price_start     = price_now
                        current_tick_start      = tick_now
                        current_sqrtPrice_start = sqrt_now

                # -----------------------------------------------------------
                # 2. If rebalance, recompute for all legs
                # -----------------------------------------------------------
                if do_rebalance:
                    rebalance_flag_arr[idx] = True

                    for i, leg in enumerate(legs):
                        rebalance_id[i] += 1

                        is_long = leg in long_options
                        is_put  = leg in put_options

                        strike = self.strike_from_delta(
                            spot_price   = spot_for_strike,
                            delta        = self.delta[leg],
                            range_factor = self.range_perc,
                            is_long      = is_long,
                            is_put       = is_put,
                        )
                        strike_tick = self.convert_price_to_tick(p=strike)

                        current_strike[i]      = strike
                        current_strike_tick[i] = strike_tick
                        current_price_a[i]     = strike / self.range_perc
                        current_price_b[i]     = strike * self.range_perc

                # -----------------------------------------------------------
                # 3. Forward-fill (write state to this row)
                # -----------------------------------------------------------
                price_start_arr[idx]     = current_price_start
                tick_start_arr[idx]      = current_tick_start
                sqrtPrice_start_arr[idx] = current_sqrtPrice_start

                for i in range(n_legs):
                    strike_start_arr[i][idx]      = current_strike[i]
                    strike_tick_start_arr[i][idx] = current_strike_tick[i]
                    price_a_arr[i][idx]           = current_price_a[i]
                    price_b_arr[i][idx]           = current_price_b[i]
                    rebalance_id_arr[i][idx]      = rebalance_id[i]

            # -----------------------------------------------------------
            # 4. Attach arrays back to df
            # -----------------------------------------------------------
            df['rebalance_flag'] = rebalance_flag_arr
            df['price_start']    = price_start_arr
            df['tick_start']     = tick_start_arr
            df['sqrtPrice_start'] = sqrtPrice_start_arr

            for i in range(n_legs):
                df[f'strike_start_{i}']      = strike_start_arr[i]
                df[f'strike_tick_start_{i}'] = strike_tick_start_arr[i]
                df[f'price_a_{i}']           = price_a_arr[i]
                df[f'price_b_{i}']           = price_b_arr[i]
                df[f'rebalance_id_{i}']      = rebalance_id_arr[i]

            self.oor = df
        """NEW-oor"""

    def calc_fees(self, s: pd.Series) -> pd.Series:
        """Calculate fee collected by us per swap
        (Uses sqrtPrice and liq_per_tick)"""
        # We ignore decimals for sqrtPrice - they end up cancelling out in calc_liq_per_tick(), calc_fees(), and get_fee_summary()
        # Our fees
        # TODO: current code assumes all legs are the same width, so fee per liquidity is the same
        # TODO: can update code to calculate fees per liquidity for differing leg widths
        low_bound = np.sqrt(self.BASE ** (s['strike_tick_start_0'] - self.tick_width))
        up_bound = np.sqrt(self.BASE ** (s['strike_tick_start_0'] + self.tick_width))
        prev_adj = np.clip(s['prev_sqrtPrice'], low_bound, up_bound)
        curr_adj = np.clip(s['sqrtPrice'], low_bound, up_bound)

        if s['change_sqrtPrice'] >= 0: # price of token 0 in token 1 went up -> fee collected in token 1
            fee_asset = self.token_1
            our_fee = s['liq_per_tick'] * (curr_adj - prev_adj) * self.fee_rate
        
        else: # price of token 0 in token 1 went down -> fee collected in token 0
            fee_asset = self.token_0
            our_fee = s['liq_per_tick'] * ((1 / curr_adj) - (1 / prev_adj)) * self.fee_rate
        return pd.Series([our_fee, fee_asset])

    def get_fee_summary(self, df: pd.DataFrame, option_type: str) -> pd.Series:
        """Get daily summary statistics of our collected fees"""
        # We ignore decimals for sqrtPrice - they end up cancelling out in calc_liq_per_tick(), calc_fees(), and get_fee_summary()
        price_close = df['price'][-1]
        sqrtPrice_close = df['sqrtPrice'][-1]

        # Our performance
        fees_0 = df.loc[df['fee_asset'] == self.token_0, 'our_fee'].sum()
        fees_1 = df.loc[df['fee_asset'] == self.token_1, 'our_fee'].sum()

        fees_total = fees_1 + fees_0 * (sqrtPrice_close ** 2) # in terms of token 1 (with dec_1)

        if self.inverse_price:
            fees_total *= price_close # in terms of token 0

        fees_total *= self.spread_mult # Fees owed to seller are multiplied by "spread" for popular strike prices
        
        short_options = ['SHORT_PUT', 'SHORT_CALL']
        long_options = ['LONG_PUT', 'LONG_CALL']
        if option_type in long_options:
            fees_total *= -1 # Fees are owed

        # fees_perc = fees_total / self.collateral

        return pd.Series([
                        fees_total,
                        ])

    def calc_position(self, df: pd.DataFrame, option_type: str, leg_number: int) -> pd.Series:
        """
        Calculates strategy position in token y, the numeraire
        (if inverse_price then this is token_0, else token_1)
        and daily returns (excluding fees).
        Expects df to be LP data of swaps.

        p_a: lower price range
        p_b: upper price range
        p_0: mid-price (initial price)
        x_0: initial amount of token x
        y_0: initial amount of token y

        p_1: price at beginning of day (same as p_0
            for daily rebalancing but different for
            weekly and monthly rebalancing)
        x_1: amount of token x at BOD
        y_1: amount of token y at BOD

        p: current (close) price at EOD
        x: current amount of token x
        y: current amount of token y
        IL: impermanent loss (loss compared to HODLing)

        """
        # Initial LP price range and positions
        p_a = df[f'price_a_{leg_number}'][-1]
        p_b = df[f'price_b_{leg_number}'][-1]
        p_0 = df['price_start'][-1]

        # Calculate liquidity factor L
        x_c = self.our_capital / df[f'strike_start_{leg_number}'][-1] # If price is below LP position, then LP position is entirely composed of (1 / strike) ETH
        y_c = self.our_capital # If price is above LP position, then LP position is entirely composed of 1 USDC
        # https://atiselsts.github.io/pdfs/uniswap-v3-liquidity-math.pdf
        L_x = x_c * ((np.sqrt(p_a) * np.sqrt(p_b)) / (np.sqrt(p_b) - np.sqrt(p_a)))
        L_y = y_c / (np.sqrt(p_b) - np.sqrt(p_a))
        L = min(L_x, L_y)
        
        # Calculate starting token X and Y amounts needed to deposit into the actual LP position based on starting price and price range
        x_0 = L * (np.sqrt(p_b) - np.sqrt(p_0)) / (np.sqrt(p_0) * np.sqrt(p_b))
        y_0 = L * (np.sqrt(p_0) - np.sqrt(p_a))
        pos_0_y = (x_0 * p_0) + y_0
        pos_0_x = (y_0 / p_0) + x_0

        # End of time period position value
        p = df['price'][-1] # close (current) price
        p_adj = np.clip(p, p_a, p_b)
        x = L * (np.sqrt(p_b) - np.sqrt(p_adj)) / (np.sqrt(p_adj) * np.sqrt(p_b))
        y = L * (np.sqrt(p_adj) - np.sqrt(p_a))
        pos_y = x * p + y # in token y
        pos_x = pos_y / p

        if option_type == 'SHORT_PUT':
            # Same as LPing
            pnl_y = pos_y - pos_0_y # in token y
        elif option_type == 'LONG_PUT':
            # Option buyer owns pos_0_y (our_capital) since LP position immediately swapped to token y when borrowed
            # Option buyer owes pos_y to option seller
            pnl_y = pos_0_y - pos_y # in token y
            
        elif option_type == 'SHORT_CALL': #TODO: use this instead: # (self.our_capital / strike) * (p_0 - p)
            # LPing but with borrowed token x
            # Short call = short put + short token x
            pnl_short_put = pos_y - pos_0_y # in token y
            pnl_short_risky_asset_y = (self.our_capital / df[f'strike_start_{leg_number}'][-1]) * (p_0 - p)
            pnl_y = pnl_short_put + pnl_short_risky_asset_y
        elif option_type == 'LONG_CALL':
            # Opposite of Short Call
            pnl_short_put = pos_y - pos_0_y # in token y
            pnl_short_risky_asset_y = (self.our_capital / df[f'strike_start_{leg_number}'][-1]) * (p_0 - p)
            pnl_y = -1 * (pnl_short_put + pnl_short_risky_asset_y)

        return pd.Series({
            'x_0': x_0,
            'y_0': y_0,
            'x': x,
            'y': y,
            'pos': pos_y,
            'pnl': pnl_y,
        })
    
    """NEW"""
    def _lp_contract_size_underlying(self, gdf: pd.DataFrame, leg_number: int = 0) -> float:
        """
        Approximate underlying units per contract for this LP-shaped option.

        We define this as the amount of token_x (the underlying, e.g. ETH)
        that the LP holds when the price is completely out of range BELOW
        the band (p <= p_a), i.e. when the position is fully converted into x.

        This uses the same capital assumptions and liquidity math as calc_position.
        """
        # Use the first row of the group as the "start-of-period" definition
        row = gdf.iloc[0]

        p_a = row[f'price_a_{leg_number}']       # lower bound
        p_b = row[f'price_b_{leg_number}']       # upper bound
        strike = row[f'strike_start_{leg_number}']

        # Capital assumptions match calc_position
        x_c = self.our_capital / strike          # if below range, LP is all token_x
        y_c = self.our_capital                   # if above range, LP is all token_y

        # Liquidity factor L (same as in calc_position)
        L_x = x_c * ((np.sqrt(p_a) * np.sqrt(p_b)) / (np.sqrt(p_b) - np.sqrt(p_a)))
        L_y = y_c / (np.sqrt(p_b) - np.sqrt(p_a))
        L = min(L_x, L_y)

        # When price is fully out-of-range BELOW the band (p <= p_a),
        # the LP holds only x:
        x_max = L * (np.sqrt(p_b) - np.sqrt(p_a)) / (np.sqrt(p_a) * np.sqrt(p_b))

        return x_max
    """NEW"""

    def convert_price(self, p: float) -> float:
        """Gets (inverse) price (convenient for pairs like USDC/ETH)
        Inverse price: price of token1 in terms of token0
        Regular price: price of token0 in terms of token1"""
        if self.inverse_price:
            return 10 ** (self.dec_1 - self.dec_0) / p
        else:
            return p / (10 ** (self.dec_1 - self.dec_0))

    def convert_tick(self, tick: int) -> float:
        """Converts tick to price"""
        return self.convert_price(self.BASE ** tick)

    def convert_price_to_tick(self, p: float) -> float:
        """Converts price to tick"""    
        return math.log(self.convert_price_reverse(p), self.BASE)

    def convert_price_reverse(self, p: float) -> float:
        """Helper function to get price for convert_price_to_tick()
        Inverse price: price of token1 in terms of token0
        Regular price: price of token0 in terms of token1"""
        if self.inverse_price:
            return 10 ** (self.dec_1 - self.dec_0) / p
        else:
            return p * (10 ** (self.dec_1 - self.dec_0))

    def calc_liq_per_tick(self, s: pd.Series) -> float:
        """
        Calculates our liquidity per tick
        (see https://atiselsts.github.io/pdfs/uniswap-v3-liquidity-math.pdf)
        """
        ''' Do not invert prices for calculating liquidity per tick
        (This is an input to self.calc_fees() which uses sqrtPrice and assumes no inverted price)
        Note that sqrtPrice should be multiplied by 10 ^ [(dec0 - dec1) / 2]
        (or alternative multiply sqrtPrice ^2 by 10 ^ (dec0 - dec1))
        to achieve correct decimal places
        But we ignore decimals here, as they end up cancelling out in anyway in calc_liq_per_tick(), calc_fees(), and get_fee_summary() '''
        price = s['sqrtPrice_start'] ** 2
        p_a = price / self.range_perc

        p_0 = s['price_start']
        if self.inverse_price:
            y_0 = (self.our_capital / p_0) / 2
        else:
            y_0 = self.our_capital / 2 # In the case of USDC this is just $0.50

        # # Initial liquidity per tick
        L = y_0 / (np.sqrt(price) - np.sqrt(p_a))
        return L

    @staticmethod
    def get_delta(spot_price, strike, range_factor, is_long, is_put):  
        return_multiplier = -1 if is_long else 1

        if spot_price < (strike / range_factor):
            return return_multiplier if is_put else 0
        elif spot_price > (strike * range_factor):
            return 0 if is_put else -1 * return_multiplier
        elif is_put:
            return return_multiplier * ((np.sqrt((strike * range_factor) / spot_price) - 1) / (range_factor - 1))
        else:
            return return_multiplier * (((np.sqrt((strike * range_factor) / spot_price) - 1) / (range_factor - 1)) - 1)

    @staticmethod
    def strike_from_delta(
        spot_price: float,
        delta: float,
        range_factor: float,
        is_long: bool,
        is_put: bool) -> float:
        """
        Inverts the original delta(...) function. Given a desired delta,
        solves for the strike that would produce that delta, under the 
        piecewise rules.

        NOTE: When delta is exactly 0 or ±1, the 'solution' is actually 
            a range of strikes. This function returns the *boundary* 
            strike (spot_price * range_factor, or spot_price/range_factor) 
            that *just* satisfies the region transition.
        """
        if (is_long and is_put) or not (is_long or is_put):
            assert(delta >= -1 and delta <= 0)
        else:
            assert(delta >= 0 and delta <= 1)

        return_multiplier = -1 if is_long else 1
        # -------------------------------
        # 1) Handle put options
        # -------------------------------
        if is_put:
            # Region 1 (put): delta = return_multiplier => => "far ITM"
            # => spot_price < strike / range_factor => i.e. strike > spot_price * range_factor
            # We'll pick boundary: strike = spot_price * range_factor.
            if delta == return_multiplier:
                # e.g. if is_long => delta = -1 => strike >= spot_price * range_factor
                # pick boundary:
                return spot_price * range_factor

            # Region 2 (put): delta = 0 => => "far OTM"
            # => spot_price > strike * range_factor => i.e. strike < spot_price / range_factor
            # We'll pick boundary: strike = spot_price / range_factor.
            elif delta == 0:
                return spot_price / range_factor

            # Region 3 (put):  0 < |delta| < 1
            # Solve:  delta = return_multiplier * ( (sqrt(...) - 1) / (range_factor - 1) )
            # eq = delta / return_multiplier
            else:
                eq = delta / return_multiplier
                x  = 1 + eq * (range_factor - 1)   # sqrt((strike * range_factor)/spot_price)

                # Guard against negative under sqrt, or negative strike
                if x <= 0:
                    raise ValueError(
                        f"Cannot invert delta={delta} in the 'middle region' for a put. "
                        f"Computed x={x} <= 0 => no real solution."
                    )
                strike = (x**2 * spot_price) / range_factor
                return strike

        # -------------------------------
        # 2) Handle call options (not is_put)
        # -------------------------------
        else:
            # Region 1 (call): delta = 0 => => "far OTM"
            # => spot_price < strike / range_factor => => strike > spot_price * range_factor
            # We'll pick boundary: strike = spot_price * range_factor
            if delta == 0:
                return spot_price * range_factor

            # Region 2 (call): delta = -1 * return_multiplier => => "far ITM"
            # => spot_price > strike * range_factor => => strike < spot_price / range_factor
            # We'll pick boundary: strike = spot_price / range_factor
            elif delta == -1 * return_multiplier:
                return spot_price / range_factor

            # Region 3 (call): solve
            #  delta = return_multiplier * ( (sqrt(...) - 1)/(range_factor - 1) - 1 )
            eq = delta / return_multiplier
            # eq + 1 = (sqrt(...) - 1)/(range_factor - 1)
            # sqrt(...)= 1 + (eq+1)*(range_factor-1)
            x  = 1 + (eq + 1)*(range_factor - 1)
            if x <= 0:
                raise ValueError(
                    f"Cannot invert delta={delta} in the 'middle region' for a call. "
                    f"Computed x={x} <= 0 => no real solution."
                )
            strike = (x**2 * spot_price) / range_factor
            return strike

    @staticmethod
    def convert_r_to_tick_width(r: float) -> float:
        """Converts range factor to tick width
        
        :r range factor
        """
        return math.log(r, 1.0001)

    @staticmethod
    def calc_sqrtprice_change(df: pd.DataFrame) -> pd.DataFrame:
        """Stores previous sqrt price and calculates delta"""
        df['prev_sqrtPrice'] = df['sqrtPrice'].shift()
        df['change_sqrtPrice'] = df['sqrtPrice'] - df['prev_sqrtPrice']
        return df

    @staticmethod
    def unpack_sqrtprice(sqrt_p: str, is_hex: bool=True) -> float:
        """Unpacks sqrt price from raw UNI V3 pool data"""
        if is_hex:
            return int(sqrt_p, 16) / (2 ** 96)
        else:
            return float(sqrt_p) / (2 ** 96)

    @staticmethod
    def get_twos_comp(hex_str: str, bits: int=256) -> float:
        """Calculate two's complement"""
        num = int(hex_str, 16)
        if (num & (1 << (bits - 1))) != 0: # Check if first bit is set
            num = num - (1 << bits)        # Get two's complement
        return num

    @staticmethod
    def get_pool_address(token_0: str, token_1: str, fee: int, chain: str='Ethereum') -> str:
        """Gets Univ3 Pool Address"""
        # Source: https://info.uniswap.org/#/pools
        # Feel free to add additional pools
        # (make sure token0 and token1 are specified in the same order as on Uniswap!)
        # Ex: '[token0]/[token1] [fee]bps': '[pool_address]',
        
        if chain == 'Ethereum':
            pools = {
                '1INCH/ETH 100bps': '0xe931b03260b2854e77e8da8378a1bc017b13cb97',
                'AAVE/ETH 30bps': '0x5ab53ee1d50eef2c1dd3d5402789cd27bb52c1bb',
                'APE/ETH 30bps': '0xac4b3dacb91461209ae9d41ec517c2b9cb1b7daf',
                'BIT/ETH 30bps': '0x5c128d25a21f681e678cb050e551a895c9309945',
                'BUSD/USDC 5bps': '0x00cef0386ed94d738c8f8a74e8bfd0376926d24c',
                'cbETH/ETH 5bps': '0x840deeef2f115cf50da625f7368c24af6fe74410',
                'DAI/ETH 5bps': '0x60594a405d53811d3bc4766596efd80fd545a270',
                'DAI/ETH 30bps': '0xc2e9f25be6257c210d7adf0d4cd6e3e881ba25f8',
                'DAI/USDC 1bps': '0x5777d92f208679db4b9778590fa3cab3ac9e2168',
                'DAI/USDC 5bps': '0x6c6bc977e13df9b0de53b251522280bb72383700',
                'DAI/FRAX 5bps': '0x97e7d56a0408570ba1a7852de36350f7713906ec',
                'ETH/BTT 30bps': '0x64a078926ad9f9e88016c199017aea196e3899e1',
                'ETH/ENS 30bps': '0x92560c178ce069cc014138ed3c2f5221ba71f58a',
                'ETH/LOOKS 30bps': '0x4b5ab61593a2401b1075b90c04cbcdd3f87ce011',
                'ETH/sETH2 30bps': '0x7379e81228514a1d2a6cf7559203998e20598346',
                'ETH/USDT 5bps': '0x11b815efb8f581194ae79006d24e0d814b7697f6',
                'ETH/USDT 30bps': '0x4e68ccd3e89f51c3074ca5072bbac773960dfa36',
                'FRAX/USDC 5bps': '0xc63b0708e2f7e69cb8a1df0e1389a98c35a76d52',
                'GNO/ETH 30bps': '0xf56d08221b5942c428acc5de8f78489a97fc5599',
                'HEX/ETH 30bps': '0x9e0905249ceefffb9605e034b534544684a58be6',
                'HEX/USDC 30bps': '0x69d91b94f0aaf8e8a2586909fa77a5c2c89818d5',
                'LDO/ETH 30bps': '0xa3f558aebaecaf0e11ca4b2199cc5ed341edfd74',
                'LINK/ETH 30bps': '0xa6cc3c2531fdaa6ae1a3ca84c2855806728693e8',
                'MATIC/ETH 30bps': '0x290a6a7460b308ee3f19023d2d00de604bcf5b42',
                'MKR/ETH 30bps': '0xe8c6c9227491c0a8156a0106a0204d881bb7e531',
                'SHIB/ETH 30bps': '0x2f62f2b4c5fcd7570a709dec05d68ea19c82a9ec',
                'UNI/ETH 30bps': '0x1d42064fc4beb5f8aaf85f4617ae8b3b5b8bd801',
                'USDC/ETH 1bps': '0xe0554a476a092703abdb3ef35c80e0d76d32939f',
                'USDC/ETH 5bps': '0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640',
                'USDC/ETH 30bps': '0x8ad599c3a0ff1de082011efddc58f1908eb6e6d8',
                'USDC/ETH 100bps': '0x7bea39867e4169dbe237d55c8242a8f2fcdcc387',
                'USDC/USDT 1bps': '0x3416cf6c708da44db2624d63ea0aaef7113527c6',
                'USDC/USDT 5bps': '0x7858e59e0c01ea06df3af3d20ac7b0003275d4bf',
                'USDC/USDM 5bps': '0x8ee3cc8e29e72e03c4ab430d7b7e08549f0c71cc',
                'WBTC/ETH 5bps': '0x4585fe77225b41b697c938b018e2ac67ac5a20c0',    
                'WBTC/ETH 30bps': '0xcbcdf9626bc03e24f779434178a73a0b4bad62ed',
                'WBTC/USDC 30bps': '0x99ac8ca7087fa4a2a1fb6357269965a2014abc35',
                'WOOF/ETH 100bps': '0x666ed8c2151f00e7e58b4d941f65a9df68d2245b',
            }
        elif chain == 'Base':
            pools = {
                'ETH/USDC 5bps': '0xd0b53d9277642d899df5c87a3966a349a798f224',
                'USDC/cbBTC 5bps': '0xfbb6eed8e7aa03b138556eedaf5d271a5e1e43ef',
                'ETH/USDC 30bps': '0x6c561b446416e1a00e8e93e221854d6ea4171372',
                'ETH/cbBTC 30bps': '0x8c7080564b5a792a33ef2fd473fba6364d5495e5',
                'ETH/BRETT 100bps': '0xba3f945812a83471d709bce9c3ca699a19fb46f7',
                'ETH/cbBTC 5bps': '0x7aea2e8a3843516afa07293a10ac8e49906dabd1',
            }

        pool_name = f"{token_0}/{token_1} {str(fee)}bps"
        try:
            pool_address = pools[pool_name]
        except KeyError:
            return None
        return pool_address

    @staticmethod
    def get_token_dec(token: str) -> int:
        """Gets number of decimals corresponding to token"""
        # Source: https://apiv5.paraswap.io/tokens/?network=1 & https://etherscan.io/tokens
        decimals = {
            '1INCH': 18,
            'AAVE': 18,
            'APE': 18,
            'BIT': 18,
            'BRETT': 18,
            'BTT': 18,
            'BUSD': 18,
            'cbETH': 18,
            'cbBTC': 8,
            'DAI': 18,
            'ENS': 18,
            'ETH': 18,
            'FRAX': 18,
            'GNO': 18,
            'HEX': 8,
            'LDO': 18,
            'LINK': 18,
            'LOOKS': 18,
            'MATIC': 18,
            'MKR': 18,
            'sETH2': 18,
            'SHIB': 18,
            'UNI': 18,
            'USDC': 6,
            'USDM': 18,
            'USDT': 6,
            'WBTC': 8,
            'WOOF': 18,
        }
        try:
            dec = decimals[token]
        except KeyError:
            return None
        return dec

    @staticmethod
    def get_tick_spacing(fee: int) -> int:
        """
        Gets Univ3 tick spacing corresponding to fee-tier

        :fee fee-tier in bps
        """
        spacing = {
            1: 1,
            5: 10,
            30: 60,
            100: 200
        }
        try:
            space = spacing[fee]
        except KeyError:
            return None
        return spacing[fee]

    def _leg_flags(self, leg_name: str):
        is_long = True if leg_name in ['LONG_PUT','LONG_CALL'] else False
        is_put = True if leg_name in ['LONG_PUT','SHORT_PUT'] else False
        return is_long, is_put

    def _net_delta(self, spot, strikes_dict):
        """Compute net delta across all legs at a given spot for current strikes."""
        net = 0.0
        for i, (leg, qty) in enumerate(self.legs.items()):
            is_long, is_put = self._leg_flags(leg)
            d = LP_Rebalance.get_delta(spot, strikes_dict[i], self.range_perc, is_long, is_put)
            net += qty * d
        return net

    def _segment_pnl(self, seg_df, strikes):
        """Sum PnL across legs for a df segment given fixed strikes."""
        seg = seg_df.copy()
        for i in range(len(self.legs)):
            seg[f'strike_start_{i}'] = strikes[i]
        pnl_total = 0.0
        for i, (leg, qty) in enumerate(self.legs.items()):
            res = self.calc_position(seg, option_type=leg, leg_number=i)
            pnl_total += qty * float(res['pnl'])
        return pnl_total


    def calc_long_call_delta_bands(
        self,
        df: pd.DataFrame,
        group_keys: list = ['date'], # downstream plots assume we group by date
        target_delta: float = 0.00,
        lower_band: float = -0.20,
        upper_band: float = 0.20
    ) -> pd.DataFrame:
        """
        Compute PnL with delta-band rebalancing for a single LONG_CALL leg.
        We trade the underlying at spot to bring (option_delta + hedge_delta) back to target_delta
        whenever total delta drifts below lower_band or above upper_band.
        """
        # Validate legs
        if not (len(self.legs) == 1 and list(self.legs.keys())[0] == 'LONG_CALL'):
            raise ValueError("calc_long_call_delta_bands expects a single LONG_CALL leg.")

        tmp = df.copy()
        if not isinstance(tmp.index, pd.DatetimeIndex):
            if 'block_timestamp' in tmp.columns:
                tmp = tmp.set_index(pd.to_datetime(tmp['block_timestamp']))
            else:
                tmp.index = pd.to_datetime(tmp.index)

        if 'price' not in tmp.columns:
            raise ValueError("DataFrame must include a 'price' column.")

        # Ensure grouping columns
        if isinstance(tmp.index, pd.DatetimeIndex):
            if 'date' not in tmp.columns:
                tmp['date'] = tmp.index.date
            iso = tmp.index.isocalendar()
            if 'year' not in tmp.columns:
                tmp['year'] = iso['year'].to_numpy()
            if 'weeknumber' not in tmp.columns:
                tmp['weeknumber'] = iso['week'].to_numpy()
            if 'month' not in tmp.columns:
                tmp['month'] = tmp.index.month

        grouped = tmp.groupby(group_keys, sort=True)
        out_rows = []

        qty = list(self.legs.values())[0]
        for gkey, gdf in grouped:
            gdf = gdf.sort_index()
            start_spot = float(gdf['price'].iloc[0])

            # === NEW: LP-based contract size (underlying units when fully out of range) ===
            contract_size_x = self._lp_contract_size_underlying(gdf, leg_number=0)
            target_delta_scaled = target_delta * contract_size_x * qty
            lower_band_scaled   = lower_band * contract_size_x * qty
            upper_band_scaled   = upper_band * contract_size_x * qty

            # Hedge state: units of underlying and cash from trades
            hedge_units = 0.0
            cash = 0.0
            num_hedges = 0

            # Iterate through time, check band breaches using total delta = option + hedge
            for i in range(len(gdf)):
                spot = float(gdf['price'].iloc[i])
                strike = float(gdf['strike_start_0'].iloc[i]) # we assume there is only one long call leg, we take the strike of the position

                # Normalized delta for ONE 1-unit-underlying call
                unit_delta = LP_Rebalance.get_delta(
                    spot_price=spot,
                    strike=strike,
                    range_factor=self.range_perc,
                    is_long=True,
                    is_put=False
                )
                # Option delta in *underlying units* for this leg:
                opt_delta = qty * unit_delta * contract_size_x

                total_delta = opt_delta + hedge_units

                if total_delta < lower_band_scaled:
                    buy_qty = target_delta_scaled - total_delta
                    if buy_qty > 0:
                        hedge_units += buy_qty
                        cash -= buy_qty * spot
                        num_hedges += 1
                elif total_delta > upper_band_scaled:
                    sell_qty = total_delta - target_delta_scaled
                    if sell_qty > 0:
                        hedge_units -= sell_qty
                        cash += sell_qty * spot
                        num_hedges += 1

            # Mark-to-market the hedge at the period end
            final_spot = float(gdf['price'].iloc[-1])
            hedge_pnl = cash + hedge_units * final_spot

            # Delta of the position at period end (option + hedge)
            opt_delta_end = qty * LP_Rebalance.get_delta(
                spot_price=final_spot,
                strike=strike,
                range_factor=self.range_perc,
                is_long=True,
                is_put=False
            ) * contract_size_x
            
            total_delta_end = opt_delta_end + hedge_units
            
            lp_rebalances = gdf['rebalance_id_0'].nunique() - 1

            """NEW-oor"""
            period_end = gdf.index[-1].date()
            """NEW-oor"""
            out_rows.append({
                'date': period_end,
                'cash': cash,
                'hedge_units': hedge_units,
                'hedge_pnl': hedge_pnl,
                'start_spot_price': start_spot,
                'end_spot_price': final_spot,
                'total_delta': total_delta_end,
                'number_of_hedges_from_threshold_breach': num_hedges,
                'number_of_rebalances': lp_rebalances
            })

        out = pd.DataFrame(out_rows).set_index('date')
        return out

### Parse Data

In [36]:
# https://lambert-guillaume.medium.com/how-to-create-a-perpetual-options-in-uniswap-v3-3c40007ccf1
def calc_DTE(r: float, volatility: float = 0.5) -> float:
    """Calculates corresponding options DTE for a UniV3 range factor"""
    return 365 * (2 * np.pi / volatility ** 2) * ( (np.sqrt(r) - 1) / (np.sqrt(r) + 1) ) ** 2

for r in [1.05, 1.25, 1.6]:
    print(f"A range factor of {r} corresponds to {calc_DTE(r)} days")

A range factor of 1.05 corresponds to 1.3646906800804546 days
A range factor of 1.25 corresponds to 28.489251631280393 days
A range factor of 1.6 corresponds to 125.49621040917174 days


In [37]:
# === Delta Bands + Rebalance back to Target Delta (Long 0.5-Delta Call) ===
local_dir = 'D:\Panoptic (ABI & MPL Files)'

TARGET_DELTA = 0.00
LOWER_BAND = -0.20
UPPER_BAND = 0.20

# --- Weekly-width strategy (12.7) ---
longcall_weekly = LP_Rebalance(
    token_0='USDC', token_1='ETH', fee=30, local_dir=local_dir,
    start_t='2024-01-01', end_t='2025-12-01',
    range_perc=12.7, col_ratio=100, com_ratio=0,
    PLP_annual_yield=0, spread_mult=1, pool_data=None,
    inverse_price=True, legs={'LONG_CALL':1}, delta={'LONG_CALL':0.5},
    chain='Ethereum'
)
"""NEW-oor"""
longcall_weekly.run_strat('oor')
"""NEW-oor"""
#longcall_weekly.daily_reb = longcall_weekly.calc_long_call_delta_bands(
     #df=longcall_weekly.daily, group_keys=['date'],
     #target_delta=TARGET_DELTA, lower_band=LOWER_BAND, upper_band=UPPER_BAND)

"""NEW-oor"""
longcall_weekly.oor_reb = longcall_weekly.calc_long_call_delta_bands(
    df=longcall_weekly.oor, group_keys=['date'],
    target_delta=TARGET_DELTA, lower_band=LOWER_BAND, upper_band=UPPER_BAND)
"""NEW-oor"""

# --- Monthly-width strategy (27) ---
#longcall_monthly = LP_Rebalance(
    #token_0='USDC', token_1='ETH', fee=30, local_dir=local_dir,
    #start_t='2024-01-01', end_t='2025-12-01',
    #range_perc=27, col_ratio=100, com_ratio=0,
    #PLP_annual_yield=0, spread_mult=1, pool_data=None,
    #inverse_price=True, legs={'LONG_CALL':1}, delta={'LONG_CALL':0.5},
    #chain='Ethereum'
#)

#longcall_monthly.run_strat('oor')

#longcall_monthly.oor_reb = longcall_monthly.calc_long_call_delta_bands(
    #df=longcall_monthly.oor, group_keys=['date'],
    #target_delta=TARGET_DELTA, lower_band=LOWER_BAND, upper_band=UPPER_BAND)

# longcall_weekly.yearly_reb = longcall_weekly.calc_long_call_delta_bands(
#     df=longcall_weekly.yearly, group_keys=['year'],
#     target_delta=TARGET_DELTA, lower_band=LOWER_BAND, upper_band=UPPER_BAND)

# # --- Daily-width strategy (4.26) ---
#longcall_daily = LP_Rebalance(
    #token_0='USDC', token_1='ETH', fee=30, local_dir=local_dir,
    #start_t='2024-01-01', end_t='2025-12-01',
    #range_perc=4.26, col_ratio=100, com_ratio=0,
    #PLP_annual_yield=0, spread_mult=1, pool_data=None,
    #inverse_price=True, legs={'LONG_CALL':1}, delta={'LONG_CALL':0.5},
    #chain='Ethereum'
#)
#longcall_daily.run_strat('oor')

#longcall_daily.oor_reb = longcall_daily.calc_long_call_delta_bands(
    #df=longcall_daily.oor, group_keys=['date'],
    #target_delta=TARGET_DELTA, lower_band=LOWER_BAND, upper_band=UPPER_BAND)

# longcall_monthly.daily_reb = longcall_monthly.calc_long_call_delta_bands(
#     df=longcall_monthly.daily, group_keys=['date'],
#     target_delta=TARGET_DELTA, lower_band=LOWER_BAND, upper_band=UPPER_BAND)

# longcall_monthly.weekly_reb = longcall_monthly.calc_long_call_delta_bands(
#     df=longcall_monthly.weekly, group_keys=['year','weeknumber'],
#     target_delta=TARGET_DELTA, lower_band=LOWER_BAND, upper_band=UPPER_BAND)

# longcall_monthly.monthly_reb = longcall_monthly.calc_long_call_delta_bands(
#     df=longcall_monthly.monthly, group_keys=['year','month'],
#     target_delta=TARGET_DELTA, lower_band=LOWER_BAND, upper_band=UPPER_BAND)

# longcall_monthly.yearly_reb = longcall_monthly.calc_long_call_delta_bands(
#     df=longcall_monthly.yearly, group_keys=['year'],
#     target_delta=TARGET_DELTA, lower_band=LOWER_BAND, upper_band=UPPER_BAND)


# # --- Yearly-width strategy (125) ---
#longcall_yearly = LP_Rebalance(
    #token_0='USDC', token_1='ETH', fee=30, local_dir=local_dir,
    #start_t='2024-01-01', end_t='2025-12-01',
    #range_perc=125, col_ratio=100, com_ratio=0,
    #PLP_annual_yield=0, spread_mult=1, pool_data=None,
    #inverse_price=True, legs={'LONG_CALL':1}, delta={'LONG_CALL':0.5},
    #chain='Ethereum'
#)
#longcall_yearly.run_strat('oor')

#longcall_yearly.oor_reb = longcall_yearly.calc_long_call_delta_bands(
    #df=longcall_yearly.oor, group_keys=['date'],
    #target_delta=TARGET_DELTA, lower_band=LOWER_BAND, upper_band=UPPER_BAND)

# longcall_yearly.daily_reb = longcall_yearly.calc_long_call_delta_bands(
#     df=longcall_yearly.daily, group_keys=['date'],
#     target_delta=TARGET_DELTA, lower_band=LOWER_BAND, upper_band=UPPER_BAND)

# longcall_yearly.weekly_reb = longcall_yearly.calc_long_call_delta_bands(
#     df=longcall_yearly.weekly, group_keys=['year','weeknumber'],
#     target_delta=TARGET_DELTA, lower_band=LOWER_BAND, upper_band=UPPER_BAND)

# longcall_yearly.monthly_reb = longcall_yearly.calc_long_call_delta_bands(
#     df=longcall_yearly.monthly, group_keys=['year','month'],
#     target_delta=TARGET_DELTA, lower_band=LOWER_BAND, upper_band=UPPER_BAND)

# longcall_yearly.yearly_reb = longcall_yearly.calc_long_call_delta_bands(
#     df=longcall_yearly.yearly, group_keys=['year'],
#     target_delta=TARGET_DELTA, lower_band=LOWER_BAND, upper_band=UPPER_BAND)

<>:2: SyntaxWarning: invalid escape sequence '\P'
<>:2: SyntaxWarning: invalid escape sequence '\P'
C:\Users\ndbur\AppData\Local\Temp\ipykernel_25812\1629363316.py:2: SyntaxWarning: invalid escape sequence '\P'
  local_dir = 'D:\Panoptic (ABI & MPL Files)'


Loading Data...
Downloading: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████|
Running Strategy


0it [00:00, ?it/s]
Step 1: 100%|███████████████████████████████████████████████████████████████| 324785/324785 [00:10<00:00, 31715.61it/s]
0it [00:00, ?it/s]
Step 2: 100%|████████████████████████████████████████████████████████████████| 324785/324785 [02:18<00:00, 2338.59it/s]
C:\Users\ndbur\AppData\Local\Temp\ipykernel_25812\3237139475.py:711: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  price_close = df['price'][-1]
C:\Users\ndbur\AppData\Local\Temp\ipykernel_25812\3237139475.py:712: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  sqrtPrice_close = df['sqrtPrice'][-1]
C:\Users\ndbur\AppData\Local\Temp\ipykernel_2581

'NEW-oor'

In [38]:
longcall_data = longcall_weekly.data.copy(deep=True)

In [39]:
import os
import matplotlib.dates as mdates
from matplotlib.ticker import PercentFormatter


# Use Panoptic dark style for all these plots
plt.style.use(r'D:\Panoptic (ABI & MPL Files)\panoptic-dark-16_9.mplstyle')

# Folder for outputs (PNGs + CSVs)
output_dir = local_dir
os.makedirs(output_dir, exist_ok=True)

def plot_and_export_longcall_stats(strategy_name: str,
                                   df_reb: pd.DataFrame,
                                   freq_label: str):
    """
    strategy_name: 'longcall_daily', etc.
    df_reb:        corresponding *_reb DataFrame for the natural frequency
    freq_label:    'daily', 'weekly', 'monthly', 'yearly'
    """
    if df_reb is None or len(df_reb) == 0:
        print(f"[WARN] No data for {strategy_name} ({freq_label}) – skipping.")
        return

    df = df_reb.copy()

    # Ensure datetime index
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)
    df = df.sort_index()

    # Ensure all needed columns exist (fill missing with NaN)
    needed_cols = [
        "cash",
        "hedge_units",
        "hedge_pnl",
        "option_pnl",
        "pnl_sum",
        "streamia_cost",
        "start_spot_price",
        "end_spot_price",
        "total_delta",
        "number_of_hedges_from_threshold_breach",
        "number_of_rebalances",
    ]
    for c in needed_cols:
        if c not in df.columns:
            df[c] = np.nan

    # --- Build the time-series plot ---
    fig, ax = plt.subplots()

    # 6) Cumulative PnL, Streamia Cost, and Net PnL
    pnl_sum = df["pnl_sum"].fillna(0)
    streamia = df["streamia_cost"].fillna(0)
    net_pnl = pnl_sum + streamia

    df["cum_net_pnl"] = net_pnl.cumsum()
   
    ax.plot(df.index, df["cum_net_pnl"])
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0))
    ax.set_ylabel("Net PnL (USDC)")
    ax.set_xlabel("Date")
    ax.set_title(f"ETH 30bps Gamma Scalping: Net PnL")

    # Match month labeling style like: Jan\n2024, Apr, Jul, Oct, Jan\n2025, ...
    locator = mdates.AutoDateLocator(minticks=6, maxticks=10)
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))
    fig.tight_layout()

    png_path = os.path.join(output_dir, f"pnl stats.png")
    fig.savefig(png_path)
    plt.close(fig)
    print(f"[INFO] Saved plot to: {png_path}")


# === Run for each longcall strategy at its natural rebalance frequency ===

"""NEW"""
assert longcall_weekly.oor_reb.index.equals(longcall_weekly.oor_fees.index), \
    "Indexes do not match!"
assert longcall_weekly.oor_reb.index.equals(longcall_weekly.oor_pos.index), \
    "Indexes do not match!"
weekly_gamma_scalping_and_streamia = longcall_weekly.oor_reb.join(longcall_weekly.oor_fees, how='left')
weekly_gamma_scalping_and_streamia.rename(columns={'fees_total': 'streamia_cost'}, inplace=True)
weekly_gamma_scalping_and_streamia = weekly_gamma_scalping_and_streamia.join(
    longcall_weekly.oor_pos['pnl'].rename('option_pnl'),
    how='left'
)
weekly_gamma_scalping_and_streamia['pnl_sum'] = (
    weekly_gamma_scalping_and_streamia['option_pnl']
    + weekly_gamma_scalping_and_streamia['hedge_pnl']
)
weekly_gamma_scalping_and_streamia['pnl_perc'] = (
    weekly_gamma_scalping_and_streamia['pnl_sum']
    / longcall_weekly.collateral
)
plot_and_export_longcall_stats(
    "longcall_weekly", weekly_gamma_scalping_and_streamia, "weekly"
)

C:\Users\ndbur\AppData\Local\Temp\ipykernel_25812\2110733278.py:70: UserWarning: The figure layout has changed to tight
  fig.tight_layout()


[INFO] Saved plot to: D:\Panoptic (ABI & MPL Files)\pnl stats.png


In [40]:
# === Long Call Strategies: Timeseries Plots + CSV Stats ===
import os

# Use Panoptic dark style for all these plots
plt.style.use(r'D:\Panoptic (ABI & MPL Files)\panoptic-dark-16_9.mplstyle')

# Folder for outputs (PNGs + CSVs)
output_dir = local_dir
os.makedirs(output_dir, exist_ok=True)

def plot_and_export_longcall_stats(strategy_name: str,
                                   df_reb: pd.DataFrame,
                                   freq_label: str):
    """
    strategy_name: 'longcall_daily', etc.
    df_reb:        corresponding *_reb DataFrame for the natural frequency
    freq_label:    'daily', 'weekly', 'monthly', 'yearly'
    """
    if df_reb is None or len(df_reb) == 0:
        print(f"[WARN] No data for {strategy_name} ({freq_label}) – skipping.")
        return

    df = df_reb.copy()

    # Ensure datetime index
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)
    df = df.sort_index()

    # Ensure all needed columns exist (fill missing with NaN)
    needed_cols = [
        "cash",
        "hedge_units",
        "hedge_pnl",
        "option_pnl",
        "pnl_sum",
        "streamia_cost",
        "start_spot_price",
        "end_spot_price",
        "total_delta",
        "number_of_hedges_from_threshold_breach",
        "number_of_rebalances",
    ]
    for c in needed_cols:
        if c not in df.columns:
            df[c] = np.nan

    # --- Build the stacked time-series plot ---
    fig, axes = plt.subplots(6, 1, sharex=True, figsize=(16, 9))

    # 1) Cash / PnL / streamia_cost (USDC terms)
    ax1 = axes[0]
    # ax1.plot(df.index, df["cash"], label="Cash")
    ax1.plot(df.index, df["hedge_pnl"], label="Hedge PnL (USDC)")
    ax1.plot(df.index, df["option_pnl"], label="Option PnL")
    ax1.plot(df.index, df["pnl_sum"], label="Total PnL")
    ax1.plot(df.index, df["streamia_cost"], label="Streamia Cost")
    ax1.set_ylabel("USDC")
    ax1.legend(loc="best", fontsize=8)
    ax1.set_title(f"{strategy_name}: PnL Components ({freq_label} hedging)")

    # 2) Total PnL vs Streamia Cost vs Net PnL
    ax2 = axes[1]
    net_pnl = (df["pnl_sum"].fillna(0) + df["streamia_cost"].fillna(0))
    ax2.plot(df.index, df["pnl_sum"], label="Total PnL")
    ax2.plot(df.index, df["streamia_cost"], label="Streamia Cost")
    ax2.plot(df.index, net_pnl, label="Net PnL (Pnl + Streamia)")
    ax2.set_ylabel("USDC")
    ax2.legend(loc="best", fontsize=8)
    ax2.set_title(f"{strategy_name}: Total vs Streamia vs Net PnL")

    # 3) Hedge units
    ax3 = axes[2]
    ax3.plot(df.index, df["hedge_units"], label="Hedge Units")
    ax3.set_ylabel("Hedge Units")
    ax3.legend(loc="best", fontsize=8)

    # 4) Spot price with rebalance markers
    ax4 = axes[3]
    ax4.plot(df.index, df["end_spot_price"], label="Spot Price", color="C0")
    ax4.set_ylabel("Spot Price")

    # Twin axis for counts
    ax4b = ax4.twinx()

    # Hedges per period
    ax4b.bar(
        df.index,
        df["number_of_hedges_from_threshold_breach"],
        width=0.8,
        alpha=0.35,
        color="C3",
        label="# Hedge Rebalances"
    )

    # LP/OOR rebalances per period
    ax4b.bar(
        df.index,
        df["number_of_rebalances"],
        width=0.4,
        alpha=0.35,
        color="C2",
        label="# LP Rebalances"
    )

    ax4b.set_ylabel("Rebalances")

    # Legends
    ax4.legend(loc="upper left", fontsize=8)
    ax4b.legend(loc="upper right", fontsize=8)

    # 5) Total delta of the position (option + hedge)
    ax5 = axes[4]
    ax5.plot(df.index, df["total_delta"], label="Total Delta")
    ax5.axhline(0.0, linestyle="--", linewidth=0.7)
    ax5.set_ylabel("Total Delta")
    ax5.set_xlabel("Date")
    ax5.legend(loc="best", fontsize=8)

    # 6) Cumulative PnL, Streamia Cost, and Net PnL
    ax6 = axes[5]
    pnl_sum = df["pnl_sum"].fillna(0)
    streamia = df["streamia_cost"].fillna(0)
    net_pnl = pnl_sum + streamia

    df["cum_pnl"] = pnl_sum.cumsum()
    df["cum_streamia"] = streamia.cumsum()
    df["cum_net_pnl"] = net_pnl.cumsum()

    ax6.plot(df.index, df["cum_pnl"], label="Cumulative Gamma Scalping PnL")
    ax6.plot(df.index, -1*df["cum_streamia"], label="Cumulative Streamia")    
    ax6.plot(df.index, df["cum_net_pnl"], label="Cumulative Net PnL")
    ax6.set_ylabel("USDC (Cumulative)")
    ax6.set_xlabel("Date")
    ax6.legend(loc="best", fontsize=8)
    ax6.set_title(f"{strategy_name}: Cumulative PnL vs Streamia vs Net PnL")

    fig.suptitle(
        f"{strategy_name} – Cash, Hedge, PnL, Spot & Delta ({freq_label} hedging)",
        fontsize=12,
    )
    fig.tight_layout()

    png_path = os.path.join(output_dir, f"{strategy_name}_{freq_label}_timeseries.png")
    fig.savefig(png_path)
    plt.close(fig)
    print(f"[INFO] Saved plot to: {png_path}")

    # --- Export CSV with per-period statistics ---
    csv_cols = [
        "cash",
        "hedge_units",
        "hedge_pnl",
        "option_pnl",
        "pnl_sum",
        "streamia_cost",
        "start_spot_price",
        "end_spot_price",
        "total_delta",
        "number_of_hedges_from_threshold_breach",
        "number_of_rebalances",
        "cum_pnl",
        "cum_streamia",
        "cum_net_pnl",
    ]

    # Include returns if present
    if "returns" in df.columns:
        csv_cols.append("returns")
    # Include pnl_perc if present (often useful)
    if "pnl_perc" in df.columns:
        csv_cols.append("pnl_perc")

    csv_path = os.path.join(output_dir, f"{strategy_name}_{freq_label}_stats.csv")
    df[csv_cols].to_csv(csv_path, index_label="date")
    print(f"[INFO] Saved CSV to: {csv_path}")


# === Run for each longcall strategy at its natural rebalance frequency ===

"""NEW"""
#assert longcall_daily.oor_reb.index.equals(longcall_daily.oor_fees.index), \
    #"Indexes do not match!"
#assert longcall_daily.oor_reb.index.equals(longcall_daily.oor_pos.index), \
    #"Indexes do not match!"
#daily_gamma_scalping_and_streamia = longcall_daily.oor_reb.join(longcall_daily.oor_fees, how='left')
#daily_gamma_scalping_and_streamia.rename(columns={'fees_total': 'streamia_cost'}, inplace=True)
#daily_gamma_scalping_and_streamia = daily_gamma_scalping_and_streamia.join(
    #longcall_daily.oor_pos['pnl'].rename('option_pnl'),
    #how='left'
#)
#daily_gamma_scalping_and_streamia['pnl_sum'] = (
    #daily_gamma_scalping_and_streamia['option_pnl']
    #+ daily_gamma_scalping_and_streamia['hedge_pnl']
#)
#daily_gamma_scalping_and_streamia['pnl_perc'] = (
    #daily_gamma_scalping_and_streamia['pnl_sum']
    #/ longcall_daily.collateral
#)
#plot_and_export_longcall_stats(
    #"longcall_daily", daily_gamma_scalping_and_streamia, "daily"
#)

assert longcall_weekly.oor_reb.index.equals(longcall_weekly.oor_fees.index), \
    "Indexes do not match!"
assert longcall_weekly.oor_reb.index.equals(longcall_weekly.oor_pos.index), \
    "Indexes do not match!"
weekly_gamma_scalping_and_streamia = longcall_weekly.oor_reb.join(longcall_weekly.oor_fees, how='left')
weekly_gamma_scalping_and_streamia.rename(columns={'fees_total': 'streamia_cost'}, inplace=True)
weekly_gamma_scalping_and_streamia = weekly_gamma_scalping_and_streamia.join(
    longcall_weekly.oor_pos['pnl'].rename('option_pnl'),
    how='left'
)
weekly_gamma_scalping_and_streamia['pnl_sum'] = (
    weekly_gamma_scalping_and_streamia['option_pnl']
    + weekly_gamma_scalping_and_streamia['hedge_pnl']
)
weekly_gamma_scalping_and_streamia['pnl_perc'] = (
    weekly_gamma_scalping_and_streamia['pnl_sum']
    / longcall_weekly.collateral
)
plot_and_export_longcall_stats(
    "longcall_weekly", weekly_gamma_scalping_and_streamia, "weekly"
)

#assert longcall_monthly.oor_reb.index.equals(longcall_monthly.oor_fees.index), \
    #"Indexes do not match!"
#assert longcall_monthly.oor_reb.index.equals(longcall_monthly.oor_pos.index), \
    #"Indexes do not match!"
#monthly_gamma_scalping_and_streamia = longcall_monthly.oor_reb.join(longcall_monthly.oor_fees, how='left')
#monthly_gamma_scalping_and_streamia.rename(columns={'fees_total': 'streamia_cost'}, inplace=True)
#monthly_gamma_scalping_and_streamia = monthly_gamma_scalping_and_streamia.join(
    #longcall_monthly.oor_pos['pnl'].rename('option_pnl'),
    #how='left'
#)
#monthly_gamma_scalping_and_streamia['pnl_sum'] = (
    #monthly_gamma_scalping_and_streamia['option_pnl']
    #+ monthly_gamma_scalping_and_streamia['hedge_pnl']
#)
#monthly_gamma_scalping_and_streamia['pnl_perc'] = (
    #monthly_gamma_scalping_and_streamia['pnl_sum']
    #/ longcall_monthly.collateral
#)
#plot_and_export_longcall_stats(
    #"longcall_monthly", monthly_gamma_scalping_and_streamia, "monthly"
#)

#assert longcall_yearly.oor_reb.index.equals(longcall_yearly.oor_fees.index), \
    #"Indexes do not match!"
#assert longcall_yearly.oor_reb.index.equals(longcall_yearly.oor_pos.index), \
    #"Indexes do not match!"
#yearly_gamma_scalping_and_streamia = longcall_yearly.oor_reb.join(longcall_yearly.oor_fees, how='left')
#yearly_gamma_scalping_and_streamia.rename(columns={'fees_total': 'streamia_cost'}, inplace=True)
#yearly_gamma_scalping_and_streamia = yearly_gamma_scalping_and_streamia.join(
    #longcall_yearly.oor_pos['pnl'].rename('option_pnl'),
    #how='left'
#)
#yearly_gamma_scalping_and_streamia['pnl_sum'] = (
    #yearly_gamma_scalping_and_streamia['option_pnl']
    #+ yearly_gamma_scalping_and_streamia['hedge_pnl']
#)
#yearly_gamma_scalping_and_streamia['pnl_perc'] = (
    #yearly_gamma_scalping_and_streamia['pnl_sum']
    #/ longcall_yearly.collateral
#)
#plot_and_export_longcall_stats(
    #"longcall_yearly", yearly_gamma_scalping_and_streamia, "yearly"
#)

C:\Users\ndbur\AppData\Local\Temp\ipykernel_25812\1300493118.py:142: UserWarning: The figure layout has changed to tight
  fig.tight_layout()


[INFO] Saved plot to: D:\Panoptic (ABI & MPL Files)\longcall_weekly_weekly_timeseries.png
[INFO] Saved CSV to: D:\Panoptic (ABI & MPL Files)\longcall_weekly_weekly_stats.csv


In [46]:
# === Weekly PnL % stats + Sharpe for Long Call w/ Streamia ===

# 1) Copy and compute weekly % returns
weekly_stats = weekly_gamma_scalping_and_streamia.copy()

# Collateral is the notional base (usually 1 for your setup)
collateral = longcall_weekly.collateral  # often == 1.0

# Gross option+hedge PnL %
weekly_stats["pnl_pct"] = weekly_stats["pnl_sum"] / collateral

# Streamia cost as % of collateral
weekly_stats["streamia_pct"] = weekly_stats["streamia_cost"] / collateral

# Net PnL after Streamia
weekly_stats["net_pnl_pct"] = (weekly_stats["pnl_sum"] + weekly_stats["streamia_cost"]) / collateral

# 2) Summary stats (min / Q1 / mean / median / Q3 / max)
def summarize(series: pd.Series, name: str):
    s = series.dropna()
    q = s.quantile([0.00, 0.25, 0.50, 0.75, 1.00])
    summary = pd.Series(
        {
            "min": q.loc[0.00],
            "Q1": q.loc[0.25],
            "mean": s.mean(),
            "median": q.loc[0.50],
            "Q3": q.loc[0.75],
            "max": q.loc[1.00],
        },
        name=name,
    )
    return summary

summary_pnl      = summarize(weekly_stats["pnl_pct"],      "gross_pnl_pct")
summary_streamia = summarize(weekly_stats["streamia_pct"], "streamia_pct")
summary_net      = summarize(weekly_stats["net_pnl_pct"],  "net_pnl_pct")

weekly_summary_table = pd.concat([summary_pnl, summary_streamia, summary_net], axis=1)
print("=== Weekly PnL % Summary (per week) ===")
print(weekly_summary_table)

# 3) Sharpe ratio (annualized) on NET weekly returns
net_weekly = weekly_stats["net_pnl_pct"].dropna()
if len(net_weekly) > 1 and net_weekly.std(ddof=1) != 0:
    sharpe_weekly = net_weekly.mean() / net_weekly.std(ddof=1)
    sharpe_annualized = sharpe_weekly * np.sqrt(52)  # 52 weeks/year
else:
    sharpe_weekly = np.nan
    sharpe_annualized = np.nan

print("\n=== Sharpe Ratio (Net PnL) ===")
print(f"Weekly Sharpe:     {sharpe_weekly:.4f}")
print(f"Annualized Sharpe: {sharpe_annualized:.4f}")

=== Weekly PnL % Summary (per week) ===
        gross_pnl_pct  streamia_pct  net_pnl_pct
min         -0.010614     -0.081853    -0.015595
Q1           0.000556     -0.003272    -0.001218
mean         0.002588     -0.002921    -0.000333
median       0.001001     -0.001916    -0.000747
Q3           0.001912     -0.001190    -0.000446
max          0.162194     -0.000151     0.080341

=== Sharpe Ratio (Net PnL) ===
Weekly Sharpe:     -0.0548
Annualized Sharpe: -0.3954


In [88]:
def plot_grouped_bars():
    import matplotlib.pyplot as plt
    import pandas as pd
    
    # Copy source dataframe
    df = weekly_gamma_scalping_and_streamia.copy()
    df = df.sort_index()

    # Daily increments of net_pnl_pct
    increments = df['pnl_perc'].astype(float) + df['streamia_cost'].astype(float)
    increments = increments.astype(float).dropna()
    # Compute skewness and kurtosis of total_returns
    from scipy.stats import skew
    from scipy.stats import kurtosis
    skewness = skew(increments, bias=False)
    kurtosis = kurtosis(increments, bias=False)
    print(f'Skewness: {skewness:.4f}')
    print(f'Excess Kurtosis: {kurtosis:.4f}')

    # Apply style
    plt.style.use(r'D:\Panoptic (ABI & MPL Files)\panoptic-dark-16_9.mplstyle')

    # Plot
    x = np.arange(len(increments))
    width = 0.6
    rotate_xticks: int = 45
    xlabel="Date"
    ylabel="Return (%)"
    title="ETH Weekly Gamma Scalping"
    fig, ax = plt.subplots(figsize=(14, 6), dpi=150)
    bars = ax.bar(x, increments * 100, width)

    # Expand x-axis limits to allow bars to border both y-axes
    ax.set_xlim(-0.5, len(x) - 0.5)

    ax.set_xticks(x)
    ax.set_xticklabels(pd.to_datetime(increments.index).strftime('%Y-%m-%d'), rotation=rotate_xticks, fontsize=10)
    ax.set_xlabel(xlabel, fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(title, fontsize=14)
    ax.tick_params(axis='y', labelsize=11)

    ax.yaxis.set_tick_params(pad=3)
    ax.spines['left'].set_position(('outward', 3))
    ax.spines['right'].set_visible(True)
    ax.yaxis.set_ticks_position('both')
    ax.tick_params(axis='y', which='both', direction='out', pad=3)
    ax.spines['right'].set_position(('axes', 1.015))

    ax.grid(True, axis='y', linestyle='--', alpha=0.7)

    for bar in bars:
        height = bar.get_height()
        if height >= 0:
            offset = 2
            va = 'bottom'
            y_pos = height
        else:
            offset = -5
            va = 'top'
            y_pos = height

        ax.annotate(f'{height:.1f}',
                    xy=(bar.get_x() + bar.get_width() / 2, y_pos),
                    xytext=(0, offset),
                    textcoords="offset points",
                    ha='center', va=va,
                    fontsize=10)

    return increments

In [ ]:
increments = plot_grouped_bars()
plt.savefig("ETH Gamma Scalping.png")

In [ ]:
from web3 import Web3
from math import sqrt

import json
import numpy as np
import pandas as pd
import pandas_gbq
import pydata_google_auth
import numpy as np
import matplotlib.pyplot as plt

pd.options.mode.chained_assignment = None
pd.set_option('display.max_columns', None)

In [11]:
class Uniswap_Pool:
    def __init__(
        self,
        token_0: str,
        token_1: str,
        fee: int, # in bps
        start_t: str,
        end_t: str,
        window: int, # in seconds
        inverse_price: bool=True,
    ) -> None:
        """
        Get Uniswap swap data and volume

        :token_0 Token 0
        :token_1 Token 1
        :fee fee rate in bps
        :start_t start time (inclusive) (i.e. '2021-05-06 00:06:12' or '2021-05-06')
        :end_t end time (exclusive)
        :inverse_price invert the price (if true: gets price of token_1 in terms of token_0)
        """
        self.token_0 = token_0
        self.token_1 = token_1
        self.fee_rate = fee / 10_000
        self.pool_address = Uniswap_Pool.get_pool_address(self.token_0, self.token_1, fee)
        self.dec_0 = Uniswap_Pool.get_token_dec(self.token_0)
        self.dec_1 = Uniswap_Pool.get_token_dec(self.token_1)
        self.tick_spacing = Uniswap_Pool.get_tick_spacing(fee)
        self.start_t = start_t
        self.end_t = end_t
        self.window = window
        self.BASE = 1.0001
        ''' If inverse_price, our numeraire is token_0
        Else, our numeraire is token_1
        Ex: For USDC/ETH pool:
        token 0 = USD
        token 1 = ETH
        Inverse_price = True -> we calculate returns in terms of USDC
        Inverse_price = False -> we calculate returns in terms of ETH
        This will affect results! '''
        self.inverse_price = inverse_price
        self.load_pool_data()

    def load_pool_data(self):
        """Loads Univ3 pool swap data"""
        # Replace this function with your own GBQ data fetcher
        # See: https://github.com/panoptic-labs/research/blob/JP/_research-bites/DataTutorial/tutorial.ipynb
        print("Loading Data...")

        SCOPES = [
            'https://www.googleapis.com/auth/cloud-platform',
            'https://www.googleapis.com/auth/drive',
        ]

        credentials = pydata_google_auth.get_user_credentials(
            SCOPES,
            auth_local_webserver=True,
        )

        query = f"""
        SELECT DISTINCT *
        FROM `arcane-world-371019.First_sync.1`
        WHERE address = '{self.pool_address}'
            AND block_timestamp >= '{self.start_t}'
            AND block_timestamp < '{self.end_t}'
        ORDER BY block_number, transaction_index
        """
        self.data = pandas_gbq.read_gbq(query, project_id = "arcane-world-371019", credentials=credentials)
        self.transform_pool_data()
    
    def transform_pool_data(self):
        """Transform amounts to human-readable format"""
        self.data['amount0'] = self.data['amount0'].apply(Uniswap_Pool.get_twos_comp)
        self.data['amount1'] = self.data['amount1'].apply(Uniswap_Pool.get_twos_comp)

        self.data['amount0'] = self.data['amount0'] / (10 ** self.dec_0)
        self.data['amount1'] = self.data['amount1'] / (10 ** self.dec_1)
        self.data['amount_total'] = self.get_swapped_amounts()

        self.data['tick_lower'] = self.nearest_multiple_of_tick_spacing(self.data['tick'])
        self.data['tick_upper'] = self.data['tick_lower'] + self.tick_spacing

        # Convert liquidity from hex to integer
        self.data['liquidity'] = self.data['liquidity'].apply(lambda x: int(x, 16))

        # Calculate liquidity for token0 and token1
        self.data['liquidity_token0'] = self.data.apply(lambda row: self.get_token0_amount( \
            row['liquidity'], row['tick'], row['tick_upper']) / (10 ** self.dec_0), axis=1)
        self.data['liquidity_token1'] = self.data.apply(lambda row: self.get_token1_amount( \
            row['liquidity'], row['tick'], row['tick_lower']) / (10 ** self.dec_1), axis=1)
        self.data['liquidity_total'] = self.get_total_tick_liquidity()
        self.data['price'] = self.data['tick'].apply(self.convert_tick)

        # Note our GBQ query ordered by blocknumber, transaciton_index
        self.data.set_index('block_timestamp', inplace=True) # Assume block timestamps are accurate "enough" for our use-case
        self.data.index = pd.to_datetime(self.data.index)

        # Get trailing swap volume
        self.data['trailing_volume'] = self.get_trailing_volume(self.window, self.data['amount_total'], self.data.index)

        # Unpack sqrt price
        self.data['sqrtPrice'] = self.data['sqrtPrice'].apply(Uniswap_Pool.unpack_sqrtprice)

        # # Calculate sqrt price change
        # self.data = Uniswap_Pool.calc_sqrtprice_change(self.data)

        self.data['date'] = self.data.index.date
        # # Lag prices (Have to do for illiquid pools w/few txs per day)
        # self.data['price_lag'] = self.data['price'].shift()
        # self.data['tick_lag'] = self.data['tick'].shift()
        # self.data['sqrtPrice_lag'] = self.data['sqrtPrice'].shift()
        # self.data = self.data.iloc[1:]

        # Calculate Uniswap pool IV timeseries
        time_threshold = self.data.index[0] + pd.Timedelta(seconds=self.window)

        # Apply the calc_iv function and set NaN values for rows before the time threshold
        self.data['iv'] = self.data.apply(lambda row: np.nan if row.name < time_threshold \
            else self.calc_iv(self.fee_rate, row['trailing_volume'], row['liquidity_total'], self.window), axis=1)
        
    def convert_price(self, p: float) -> float:
        """Gets (inverse) price (convenient for pairs like USDC/ETH)
        Inverse price: price of token1 in terms of token0
        Regular price: price of token0 in terms of token1"""
        if self.inverse_price:
            return 10 ** (self.dec_1 - self.dec_0) / p
        else:
            return p / (10 ** (self.dec_1 - self.dec_0))

    def convert_tick(self, tick: int) -> float:
        """Converts tick to price"""
        return self.convert_price(self.BASE ** tick)

    def nearest_multiple_of_tick_spacing(self, tick: int) -> int:
        """Finds the nearest multiple of the tick spacing"""
        return (tick // self.tick_spacing) * self.tick_spacing
    
    def get_swapped_amounts(self) -> pd.Series:
        """Takes absolute value of all swapped amounts in terms of quote token
        Negative (token out) becomes positive
        Positive (token in) stays the same
        """
        if self.inverse_price:
            return abs(self.data['amount0']) # token0 is quote
        else:
            return abs(self.data['amount1']) # token1 is quote        
    
    def get_total_tick_liquidity(self) -> pd.Series:
        """Gets the current tick's total liquidity in terms of either token0 or token1"""

        if self.inverse_price:
            return self.data.apply(lambda row: self.get_total_token0_amount( \
                self.fee_rate, row['liquidity'], (row['tick_upper'] + row['tick_lower']) / 2), axis=1) \
                / (10 ** self.dec_0)
        else:
            return self.data.apply(lambda row: self.get_total_token1_amount( \
                self.fee_rate, row['liquidity'], (row['tick_upper'] + row['tick_lower']) / 2), axis=1) \
                / (10 ** self.dec_1)

    @staticmethod
    def tick_to_price(tick: int):
        """Calculate price from tick"""
        return 1.0001 ** tick

    @staticmethod
    def get_trailing_volume(window: int, amounts: pd.Series, ts: pd.Series) -> pd.Series:
        """Adds up past swap volume
        :window the trailing window (in seconds) to accumulate
        :swap amounts, must be of the same token type (either token0 or token1)
        :ts timestamps corresponding with swap amounts
        """        
        trailing_volumes = []
        window_timedelta = pd.Timedelta(seconds=window)
        
        for i in range(len(ts)):
            current_time = ts[i]
            start_index = ts.searchsorted(current_time - window_timedelta, side='left')
            trailing_volume = amounts[start_index:i+1].sum()
            trailing_volumes.append(trailing_volume)
        return pd.Series(trailing_volumes, index=amounts.index)

    @staticmethod
    def calc_iv(fee_rate: float, volume: float, tickLiq: float, window: int):
        """Calculates the IV of the pool based on Guillaume's formula https://panoptic.xyz/research/new-formulation-implied-volatility#step-5-derive-the-implied-volatility-iv
        
        :fee_rate Uniswap pool fee tier
        :volume the total volume of swapped amounts in the past window
        :tickLiq the current tick's total liquidity
        :window the number of seconds we are using to lookback at volume"""
        num_secs_in_year = 60*60*24*365
        annualization_factor = num_secs_in_year / window
        return 2 * fee_rate * np.sqrt(volume / tickLiq) * np.sqrt(annualization_factor)

    @staticmethod
    def get_token0_amount(liquidity: int, tick_current: int, tick_upper: int):
        """Gets the token0 contribution toward current tick's liquidity
       
        :liquidity the raw liquidity value from Uniswap's onchain pool data
        :tick_current the current tick of the Uniswap pool
        :tick_upper the upper tick of the liquidity chunk (corresponds to the lower price of the LP position)"""
        return (liquidity) * (1 / sqrt(Uniswap_Pool.tick_to_price(tick_current)) - 1 / sqrt(Uniswap_Pool.tick_to_price(tick_upper)))

    @staticmethod
    def get_token1_amount(liquidity: int, tick_current: int, tick_lower: int):
        """Gets the token1 contribution toward current tick's liquidity
        
        :liquidity the raw liquidity value from Uniswap's onchain pool data
        :tick_current the current tick of the Uniswap pool
        :tick_lower the lower tick of the liquidity chunk (corresponds to the upper price of the LP position)"""
        return liquidity * (sqrt(Uniswap_Pool.tick_to_price(tick_current)) - sqrt(Uniswap_Pool.tick_to_price(tick_lower)))
    
    @staticmethod
    def get_total_token0_amount(fee_rate: float, liquidity: int, tick_mid: int):
        """Gets the current tick's total liquidity, but converted to token0
        
        :fee_rate the pool's fee tier
        :liquidity the raw liquidity value from Uniswap's onchain pool data
        :tick_mid the strike price of the liquidity chunk"""
        return (liquidity * fee_rate) / np.sqrt(Uniswap_Pool.tick_to_price(tick_mid))

    @staticmethod
    def get_total_token1_amount(fee_rate: float, liquidity: int, tick_mid: int):
        """Gets the current tick's total liquidity, but converted to token1
        
        :fee_rate the pool's fee tier
        :liquidity the raw liquidity value from Uniswap's onchain pool data
        :tick_mid the strike price of the liquidity chunk"""
        return liquidity * fee_rate * np.sqrt(Uniswap_Pool.tick_to_price(tick_mid))    

    @staticmethod
    def get_twos_comp(hex_str: str, bits: int=256) -> float:
        """Calculate two's complement"""
        num = int(hex_str, 16)
        if (num & (1 << (bits - 1))) != 0: # Check if first bit is set
            num = num - (1 << bits)        # Get two's complement
        return num
    
    @staticmethod
    def unpack_sqrtprice(sqrt_p: str) -> float:
        """Unpacks sqrt price from raw UNI V3 pool data"""
        return int(sqrt_p, 16) / (2 ** 96)
    
    @staticmethod
    def calc_sqrtprice_change(df: pd.DataFrame) -> pd.DataFrame:
        """Stores previous sqrt price and calculates delta"""
        df['prev_sqrtPrice'] = df['sqrtPrice'].shift()
        df['change_sqrtPrice'] = df['sqrtPrice'] - df['prev_sqrtPrice']
        return df
        
    @staticmethod
    def get_pool_address(token_0: str, token_1: str, fee: int) -> str:
        """Gets Univ3 Pool Address"""
        # Source: https://info.uniswap.org/#/pools
        # Feel free to add additional pools
        # (make sure token0 and token1 are specified in the same order as on Uniswap!)
        # Ex: '[token0]/[token1] [fee]bps': '[pool_address]',
        pools = {
            '1INCH/ETH 100bps': '0xe931b03260b2854e77e8da8378a1bc017b13cb97',
            'AAVE/ETH 30bps': '0x5ab53ee1d50eef2c1dd3d5402789cd27bb52c1bb',
            'APE/ETH 30bps': '0xac4b3dacb91461209ae9d41ec517c2b9cb1b7daf',
            'BIT/ETH 30bps': '0x5c128d25a21f681e678cb050e551a895c9309945',
            'BUSD/USDC 5bps': '0x00cef0386ed94d738c8f8a74e8bfd0376926d24c',
            'cbETH/ETH 5bps': '0x840deeef2f115cf50da625f7368c24af6fe74410',
            'DAI/ETH 5bps': '0x60594a405d53811d3bc4766596efd80fd545a270',
            'DAI/ETH 30bps': '0xc2e9f25be6257c210d7adf0d4cd6e3e881ba25f8',
            'DAI/USDC 1bps': '0x5777d92f208679db4b9778590fa3cab3ac9e2168',
            'DAI/USDC 5bps': '0x6c6bc977e13df9b0de53b251522280bb72383700',
            'DAI/FRAX 5bps': '0x97e7d56a0408570ba1a7852de36350f7713906ec',
            'ETH/BTT 30bps': '0x64a078926ad9f9e88016c199017aea196e3899e1',
            'ETH/ENS 30bps': '0x92560c178ce069cc014138ed3c2f5221ba71f58a',
            'ETH/LOOKS 30bps': '0x4b5ab61593a2401b1075b90c04cbcdd3f87ce011',
            'ETH/sETH2 30bps': '0x7379e81228514a1d2a6cf7559203998e20598346',
            'ETH/USDT 5bps': '0x11b815efb8f581194ae79006d24e0d814b7697f6',
            'ETH/USDT 30bps': '0x4e68ccd3e89f51c3074ca5072bbac773960dfa36',
            'FRAX/USDC 5bps': '0xc63b0708e2f7e69cb8a1df0e1389a98c35a76d52',
            'GNO/ETH 30bps': '0xf56d08221b5942c428acc5de8f78489a97fc5599',
            'HEX/ETH 30bps': '0x9e0905249ceefffb9605e034b534544684a58be6',
            'HEX/USDC 30bps': '0x69d91b94f0aaf8e8a2586909fa77a5c2c89818d5',
            'LDO/ETH 30bps': '0xa3f558aebaecaf0e11ca4b2199cc5ed341edfd74',
            'LINK/ETH 30bps': '0xa6cc3c2531fdaa6ae1a3ca84c2855806728693e8',
            'MATIC/ETH 30bps': '0x290a6a7460b308ee3f19023d2d00de604bcf5b42',
            'MKR/ETH 30bps': '0xe8c6c9227491c0a8156a0106a0204d881bb7e531',
            'SHIB/ETH 30bps': '0x2f62f2b4c5fcd7570a709dec05d68ea19c82a9ec',
            'UNI/ETH 30bps': '0x1d42064fc4beb5f8aaf85f4617ae8b3b5b8bd801',
            'USDC/ETH 1bps': '0xe0554a476a092703abdb3ef35c80e0d76d32939f',
            'USDC/ETH 5bps': '0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640',
            'USDC/ETH 30bps': '0x8ad599c3a0ff1de082011efddc58f1908eb6e6d8',
            'USDC/ETH 100bps': '0x7bea39867e4169dbe237d55c8242a8f2fcdcc387',
            'USDC/USDT 1bps': '0x3416cf6c708da44db2624d63ea0aaef7113527c6',
            'USDC/USDT 5bps': '0x7858e59e0c01ea06df3af3d20ac7b0003275d4bf',
            'USDC/USDM 5bps': '0x8ee3cc8e29e72e03c4ab430d7b7e08549f0c71cc',
            'WBTC/ETH 5bps': '0x4585fe77225b41b697c938b018e2ac67ac5a20c0',    
            'WBTC/ETH 30bps': '0xcbcdf9626bc03e24f779434178a73a0b4bad62ed',
            'WBTC/USDC 30bps': '0x99ac8ca7087fa4a2a1fb6357269965a2014abc35',
            'WOOF/ETH 100bps': '0x666ed8c2151f00e7e58b4d941f65a9df68d2245b',
        }

        pool_name = f"{token_0}/{token_1} {str(fee)}bps"
        try:
            pool_address = pools[pool_name]
        except KeyError:
            return None
        return pool_address

    @staticmethod
    def get_token_dec(token: str) -> int:
        """Gets number of decimals corresponding to token"""
        # Source: https://apiv5.paraswap.io/tokens/?network=1 & https://etherscan.io/tokens
        decimals = {
            '1INCH': 18,
            'AAVE': 18,
            'APE': 18,
            'BIT': 18,
            'BTT': 18,
            'BUSD': 18,
            'cbETH': 18,
            'DAI': 18,
            'ENS': 18,
            'ETH': 18,
            'FRAX': 18,
            'GNO': 18,
            'HEX': 8,
            'LDO': 18,
            'LINK': 18,
            'LOOKS': 18,
            'MATIC': 18,
            'MKR': 18,
            'sETH2': 18,
            'SHIB': 18,
            'UNI': 18,
            'USDC': 6,
            'USDM': 18,
            'USDT': 6,
            'WBTC': 8,
            'WOOF': 18,
        }
        try:
            dec = decimals[token]
        except KeyError:
            return None
        return dec

    @staticmethod
    def get_tick_spacing(fee: int) -> int:
        """
        Gets Univ3 tick spacing corresponding to fee-tier

        :fee fee-tier in bps
        """
        spacing = {
            1: 1,
            5: 10,
            30: 60,
            100: 200
        }
        try:
            space = spacing[fee]
        except KeyError:
            return None
        return spacing[fee]

In [12]:
volume = Uniswap_Pool(
            token_0 = 'USDC',
            token_1 = 'ETH',
            fee = 30,
            start_t = '2024-01-01',
            end_t = '2025-11-01',
            window = 60 * 60 * 24,
            inverse_price = True,
)

(volume.data.iv * 100).describe()

Loading Data...
Downloading: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████|


count    299232.000000
mean         76.088205
std          51.690118
min           1.981689
25%          49.525260
50%          64.058655
75%          85.448154
max         670.853511
Name: iv, dtype: float64

In [13]:
# Extract and clean the relevant time series
df = volume.data.copy()
df = df[['price', 'iv']].dropna()

# Resample to daily frequency using the last available value each day
daily = df.resample('1D').last()

# Compute daily simple returns for spot price
daily['price_return'] = daily['price'].pct_change()
daily['iv_change'] = daily['iv'].pct_change()

# Drop NaNs after computing changes
daily.dropna(inplace=True)

In [ ]:
def _compose_total_returns_like_plot(daily, daily_gamma_scalping_and_streamia):
    import pandas as pd
    import matplotlib.pyplot as plt

    plt.style.use(r'D:\Panoptic (ABI & MPL Files)\panoptic-dark-16_9.mplstyle')

    df = daily_gamma_scalping_and_streamia.copy()

    # Fix timezone issues: force both indexes to be tz-naive
    if hasattr(daily.index, "tz") and daily.index.tz is not None:
        daily.index = daily.index.tz_convert(None)

    if hasattr(df.index, "tz") and df.index.tz is not None:
        df.index = df.index.tz_convert(None)

    # Compute strategy return
    df["net_pnl_increment"] = df["pnl_perc"] + df["streamia_cost"]

    # Safe align without timezone conflict
    aligned = (
        daily[["price_return"]]
        .join(df[["net_pnl_increment"]], how="inner")
        .dropna()
    )

    # Rolling 30-day correlation
    rolling_corr = (
        aligned["price_return"]
        .rolling(30)
        .corr(aligned["net_pnl_increment"])
    )
    corr = aligned["price_return"].corr(aligned["net_pnl_increment"])
    print(f"ETH Long Gamma Scalping Correlation: {corr:.4f}")
    # Plot (simple, no explicit colors)
    plt.style.use('D:\Panoptic (ABI & MPL Files)\panoptic-dark-16_9.mplstyle')
    plt.plot(rolling_corr.index, rolling_corr.values, label='30D rolling corr')
    plt.title('Rolling 30-Day Correlation: ETH Gamma Scalping vs. Spot Returns')
    plt.xlabel('Date')
    plt.ylabel('Correlation')
    plt.xticks(fontsize=4)
    plt.yticks(np.arange(-1.0, 1.5, 0.5))